# Product Detail Ocr Ai

Parses saved product-detail page source, downloads detail images, runs OCR, and uses the Gemini API to turn the extracted text into structured product attributes (ingredients, claims, specs) added to the product table.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


> ⚠️ **Read before running.** This notebook downloads product-detail images from Coupang for OCR. Coupang's terms may restrict automated collection, and the images are the sellers' copyrighted content.
>
> It is shared to show the collection and parsing method, not as a tool to run against Coupang's image servers. Do not run it against the live site without that site's permission. To follow the downstream steps, use the synthetic files in [`sample-data/`](../sample-data/).

## 📌 파일 준비 및 작업 방법

### 1. 상품 Excel
**이전 상품 분석 코드에서 생성된 상품 Excel 결과 파일**을 사용합니다.

- `제품분석` 시트가 포함된 Excel
- Cell 2에서 다시 업로드합니다.

### 2. Coupang 상세페이지 HTML
**상품별 상세페이지를 직접 페이지 소스로 저장**해야 합니다.

1. Coupang 상품 상세페이지 접속
2. **Ctrl + U** → 페이지 소스 열기
3. **Ctrl + S** → HTML 파일로 저장
4. 상품마다 위 과정을 반복

> ⚠️ 일반 상세페이지 저장본이 아니라, 반드시 **Ctrl + U로 연 페이지 소스**를 저장해야 합니다.

### 3. Cell 2에서 업로드
Cell 2 실행 후 다음 파일을 **한 번에 업로드**합니다.

- **상품 Excel 1개**
- **상품별 상세페이지 HTML 여러 개**

### 4. 전체 작업 흐름

**상품 Excel**  
→ `제품분석`  
→ **Product Master 생성**

**상품별 HTML**  
→ 상세페이지 분석  
→ 상세 이미지 추출  
→ 이미지 다운로드  
→ OCR  
→ Gemini AI 상품정보 통합

→ 최종적으로 **기존 제품분석에 AI 상품정보 추가**

### 5. 최종 결과
기존 상품 데이터는 유지하고, AI 분석 결과를 별도 컬럼으로 추가합니다.

예:
`AI상품명` / `구성품` / `제조사` / `수입원` / `원재료` / `알레르기` / `마케팅소구포인트` / `AI사이즈` / `AI용량` / `개당중량` / `총중량_AI` / `수량_AI` / `포장재질` / `보관방법` / `소비기한` / `사용방법` 등

**확인할 수 없는 정보는 임의로 추정하지 않고 `미확인`으로 처리합니다.**

In [ ]:
# ============================================================
# [CELL 1] 환경설정 + 라이브러리 + 상품목록 Drive 자동 불러오기
#             + HTML 파일 직접 업로드
# ============================================================

# ------------------------------------------------------------
# 1. 라이브러리 설치
# ------------------------------------------------------------

!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-kor tesseract-ocr-eng
!pip -q install openpyxl pandas requests beautifulsoup4 pillow pytesseract

import os
import re
import html
import glob
import shutil
import time
import warnings
import requests

import pandas as pd
import numpy as np
import pytesseract

from PIL import Image, ImageOps, ImageEnhance
from bs4 import BeautifulSoup
from urllib.parse import unquote

from google.colab import files
from google.colab import drive

warnings.filterwarnings("ignore")


# ============================================================
# 2. Google Drive 연결
# ============================================================

print("=" * 70)
print("☁️ Google Drive 연결")
print("=" * 70)

drive.mount(
    "/content/drive",
    force_remount=False
)

print("✅ Google Drive 연결 완료")
print()


# ============================================================
# 3. 기본 경로
# ============================================================

BASE_DIR = "/content/coupang_ocr"

HTML_DIR = os.path.join(
    BASE_DIR,
    "html"
)

IMAGE_DIR = os.path.join(
    BASE_DIR,
    "images"
)

RESULT_DIR = os.path.join(
    BASE_DIR,
    "results"
)


for d in [
    BASE_DIR,
    HTML_DIR,
    IMAGE_DIR,
    RESULT_DIR
]:
    os.makedirs(
        d,
        exist_ok=True
    )


# ============================================================
# 4. Google Drive 상품목록 경로
# ============================================================

DRIVE_PRODUCT_DIR = (
    "/content/drive/MyDrive/"
    "YOUR_PROJECT_FOLDER/data_coupang"
)


print("=" * 70)
print("📊 상품목록 Excel 확인")
print("=" * 70)

print(
    f"Drive 경로 : {DRIVE_PRODUCT_DIR}"
)

if not os.path.exists(
    DRIVE_PRODUCT_DIR
):

    raise FileNotFoundError(
        "\n"
        "❌ 상품목록 Drive 폴더를 찾을 수 없습니다.\n\n"
        f"확인 경로:\n"
        f"{DRIVE_PRODUCT_DIR}"
    )

print("✅ Drive 상품목록 폴더 확인")
print()


# ============================================================
# 5. Drive에서 상품목록 Excel 자동 탐색
# ============================================================

drive_excel_files = []

for filename in os.listdir(
    DRIVE_PRODUCT_DIR
):

    full_path = os.path.join(
        DRIVE_PRODUCT_DIR,
        filename
    )

    if (
        os.path.isfile(full_path)
        and filename.lower().endswith(
            (".xlsx", ".xls")
        )
    ):

        drive_excel_files.append(
            full_path
        )


print(
    f"📑 Drive Excel 파일 : "
    f"{len(drive_excel_files):,}개"
)


if not drive_excel_files:

    raise FileNotFoundError(
        "\n"
        "❌ Drive 폴더에 Excel 파일이 없습니다.\n\n"
        f"경로:\n"
        f"{DRIVE_PRODUCT_DIR}"
    )


# ============================================================
# 6. 제품분석 시트가 있는 파일 우선 선택
# ============================================================

product_candidates = []

for path in drive_excel_files:

    try:

        xls_check = pd.ExcelFile(
            path
        )

        if "제품분석" in xls_check.sheet_names:

            product_candidates.append(
                path
            )

    except Exception:
        pass


if product_candidates:

    # 가장 최근 수정된 파일
    PRODUCT_MASTER_DRIVE_FILE = max(
        product_candidates,
        key=os.path.getmtime
    )

else:

    # 제품분석 시트가 없으면 가장 최근 Excel
    PRODUCT_MASTER_DRIVE_FILE = max(
        drive_excel_files,
        key=os.path.getmtime
    )


# ============================================================
# 7. 상품목록 Excel을 Colab으로 복사
# ============================================================

PRODUCT_MASTER_LOCAL_FILE = os.path.join(
    BASE_DIR,
    os.path.basename(
        PRODUCT_MASTER_DRIVE_FILE
    )
)


shutil.copy2(
    PRODUCT_MASTER_DRIVE_FILE,
    PRODUCT_MASTER_LOCAL_FILE
)


print()
print("✅ 상품목록 자동 불러오기 완료")

print(
    f"   Drive : "
    f"{PRODUCT_MASTER_DRIVE_FILE}"
)

print(
    f"   Local : "
    f"{PRODUCT_MASTER_LOCAL_FILE}"
)

print()


# ============================================================
# 8. 기존 HTML 폴더 초기화
#
# 매 실행마다 새 HTML을 업로드하므로
# 이전 HTML이 섞이지 않도록 삭제
# ============================================================

print("=" * 70)
print("🧹 기존 HTML 파일 정리")
print("=" * 70)


old_html_files = []

old_html_files += glob.glob(
    os.path.join(
        HTML_DIR,
        "*.html"
    )
)

old_html_files += glob.glob(
    os.path.join(
        HTML_DIR,
        "*.htm"
    )
)


for path in old_html_files:

    try:

        os.remove(path)

    except Exception:
        pass


print(
    f"🗑️ 기존 HTML 삭제 : "
    f"{len(old_html_files):,}개"
)

print()


# ============================================================
# 9. HTML 직접 업로드
# ============================================================

print("=" * 70)
print("🌐 쿠팡 HTML 파일 업로드")
print("=" * 70)

print()
print("📂 쿠팡 검색결과 HTML 파일을 모두 업로드하세요.")
print("   ※ HTML은 Google Drive에서 가져오지 않습니다.")
print()


uploaded = files.upload()


# ============================================================
# 10. HTML 업로드 결과 확인
# ============================================================

html_uploaded_files = []

for filename in uploaded.keys():

    if filename.lower().endswith(
        (".html", ".htm")
    ):

        html_uploaded_files.append(
            filename
        )


if not html_uploaded_files:

    raise FileNotFoundError(
        "\n"
        "❌ HTML 파일이 업로드되지 않았습니다.\n"
        "쿠팡 검색결과 HTML 파일을 업로드해주세요."
    )


# ============================================================
# 11. HTML을 작업 폴더로 복사
# ============================================================

copied_html_files = []


for filename in html_uploaded_files:

    src = os.path.abspath(
        filename
    )

    dst = os.path.join(
        HTML_DIR,
        filename
    )

    shutil.copy2(
        src,
        dst
    )

    copied_html_files.append(
        dst
    )


# ============================================================
# 12. 테스트 설정
# ============================================================

# 5   = 처음 5개만 테스트
# None = 전체 상품 처리

TEST_LIMIT = 5


# ============================================================
# 13. OCR 설정
# ============================================================

OCR_LANG = "kor+eng"

OCR_CONFIG = "--psm 6"


# ============================================================
# 14. 이미지 다운로드 설정
# ============================================================

REQUEST_TIMEOUT = 30

# 너무 작은 이미지는 제외
MIN_IMAGE_WIDTH = 1200


# ============================================================
# 15. 최종 환경 확인
# ============================================================

print()
print("=" * 70)
print("✅ OCR 환경설정 완료")
print("=" * 70)

print()
print("📊 상품목록")

print(
    f"   Drive 원본 : "
    f"{PRODUCT_MASTER_DRIVE_FILE}"
)

print(
    f"   Local 파일 : "
    f"{PRODUCT_MASTER_LOCAL_FILE}"
)

print()
print("🌐 HTML")

print(
    f"   업로드 파일 : "
    f"{len(copied_html_files):,}개"
)

for i, path in enumerate(
    copied_html_files,
    1
):

    print(
        f"   [{i}] "
        f"{os.path.basename(path)}"
    )

print()
print("📂 작업 폴더")

print(
    f"   BASE_DIR   : {BASE_DIR}"
)

print(
    f"   HTML_DIR   : {HTML_DIR}"
)

print(
    f"   IMAGE_DIR  : {IMAGE_DIR}"
)

print(
    f"   RESULT_DIR : {RESULT_DIR}"
)

print()
print("⚙️ OCR 설정")

print(
    f"   TEST_LIMIT : {TEST_LIMIT}"
)

print(
    f"   OCR_LANG   : {OCR_LANG}"
)

print(
    f"   OCR_CONFIG : {OCR_CONFIG}"
)

print(
    f"   MIN_IMAGE_WIDTH : "
    f"{MIN_IMAGE_WIDTH}px"
)

print()
print("=" * 70)

## **CELL 2 — Excel + HTML 업로드 / Product Master 생성**

In [ ]:
# ============================================================
# [CELL 2] Product Master 생성
#          ※ Excel / HTML 업로드는 CELL 1에서 이미 완료
# ============================================================

from IPython.display import display

import os
import pandas as pd


# ============================================================
# 1. 기존 변수 초기화
# ============================================================

ocr_master = None

html_records = []
html_df = pd.DataFrame()

matched_html = []
unmatched_html = []

ocr_results = []


# ============================================================
# 2. CELL 1 변수 확인
# ============================================================

required_variables = [
    "PRODUCT_MASTER_LOCAL_FILE",
    "HTML_DIR",
    "IMAGE_DIR",
    "RESULT_DIR",
    "TEST_LIMIT"
]


missing_variables = [
    var
    for var in required_variables
    if var not in globals()
]


if missing_variables:

    raise RuntimeError(
        "❌ CELL 1이 먼저 실행되어야 합니다.\n\n"
        f"누락된 변수: {missing_variables}"
    )


if not os.path.exists(
    PRODUCT_MASTER_LOCAL_FILE
):

    raise FileNotFoundError(
        "❌ Product Master Excel을 찾을 수 없습니다.\n\n"
        f"{PRODUCT_MASTER_LOCAL_FILE}"
    )


# ============================================================
# 3. Excel 확인
# ============================================================

print("=" * 70)
print("📊 Product Master Excel 불러오기")
print("=" * 70)

print(
    f"파일 : {PRODUCT_MASTER_LOCAL_FILE}"
)


# ============================================================
# 4. Excel 읽기
# ============================================================

xls = pd.ExcelFile(
    PRODUCT_MASTER_LOCAL_FILE
)

print()
print(
    "📄 시트 목록:",
    xls.sheet_names
)


if "제품분석" not in xls.sheet_names:

    raise ValueError(
        "❌ '제품분석' 시트를 찾을 수 없습니다.\n\n"
        f"현재 시트: {xls.sheet_names}"
    )


master = pd.read_excel(
    PRODUCT_MASTER_LOCAL_FILE,
    sheet_name="제품분석"
)


print(
    f"\n원본 상품 수 : {len(master):,}"
)


# ============================================================
# 5. 컬럼 자동 탐색
# ============================================================

def find_column(
    df,
    candidates
):

    for candidate in candidates:

        if candidate in df.columns:

            return candidate

    return None


url_col = find_column(
    master,
    [
        "상품URL",
        "상품 URL",
        "URL",
        "url",
        "쿠팡URL",
        "쿠팡 URL"
    ]
)


brand_col = find_column(
    master,
    [
        "브랜드",
        "Brand",
        "brand"
    ]
)


name_col = find_column(
    master,
    [
        "상품명",
        "상품명 ",
        "Product Name",
        "product_name",
        "제품명"
    ]
)


# ============================================================
# 6. 필수 컬럼 확인
# ============================================================

if not url_col:

    raise ValueError(
        "❌ 상품URL 컬럼을 찾지 못했습니다.\n\n"
        f"현재 컬럼:\n{list(master.columns)}"
    )


if not brand_col:

    raise ValueError(
        "❌ 브랜드 컬럼을 찾지 못했습니다."
    )


if not name_col:

    raise ValueError(
        "❌ 상품명 컬럼을 찾지 못했습니다."
    )


# ============================================================
# 7. 통합제품ID
# ============================================================

if "통합제품ID" not in master.columns:

    master.insert(
        0,
        "통합제품ID",
        [
            f"HW_{i:05d}"
            for i in range(
                len(master)
            )
        ]
    )


# ============================================================
# 8. Product Master 표준화
# ============================================================

ocr_master = master.copy()


ocr_master["상품URL"] = (
    ocr_master[url_col]
    .fillna("")
    .astype(str)
)


ocr_master["브랜드"] = (
    ocr_master[brand_col]
    .fillna("")
    .astype(str)
)


ocr_master["상품명"] = (
    ocr_master[name_col]
    .fillna("")
    .astype(str)
)


# ============================================================
# 9. itemId 표준화
# ============================================================

if "itemId" in ocr_master.columns:

    ocr_master["itemId"] = (
        ocr_master["itemId"]
        .fillna("")
        .astype(str)
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

else:

    ocr_master["itemId"] = ""


# ============================================================
# 10. vendorItemId 표준화
# ============================================================

if "vendorItemId" in ocr_master.columns:

    ocr_master["vendorItemId"] = (
        ocr_master["vendorItemId"]
        .fillna("")
        .astype(str)
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

else:

    ocr_master["vendorItemId"] = ""


# ============================================================
# 11. TEST_LIMIT
# ============================================================

if TEST_LIMIT is not None:

    ocr_master = (
        ocr_master
        .head(TEST_LIMIT)
        .copy()
    )


# ============================================================
# 12. HTML 파일 확인
# ============================================================

html_files = []

for filename in os.listdir(
    HTML_DIR
):

    if filename.lower().endswith(
        (".html", ".htm")
    ):

        html_files.append(
            os.path.join(
                HTML_DIR,
                filename
            )
        )


# ============================================================
# 13. 결과 확인
# ============================================================

print()
print("=" * 70)
print("✅ CELL 2 완료")
print("=" * 70)

print()
print(
    f"Product Master : {len(ocr_master):,}"
)

print(
    f"HTML 파일      : {len(html_files):,}"
)

print(
    f"TEST_LIMIT     : {TEST_LIMIT}"
)

print()
print(
    f"Excel : "
    f"{os.path.basename(PRODUCT_MASTER_LOCAL_FILE)}"
)

print(
    f"HTML_DIR : {HTML_DIR}"
)

print()


# ============================================================
# 14. Product Master 미리보기
# ============================================================

preview_cols = [
    "통합제품ID",
    "브랜드",
    "상품명",
    "itemId",
    "vendorItemId",
    "상품URL"
]


preview_cols = [
    col
    for col in preview_cols
    if col in ocr_master.columns
]


display(
    ocr_master[
        preview_cols
    ].head(10)
)

## **CELL 3 — HTML 분석 / itemId / vendorItemId / 상세 이미지 추출**

In [ ]:
# ============================================================
# CELL 3
# 🔍 쿠팡 HTML 분석 + 실제 상품 상세 이미지 추출
#
# - view-source HTML 복원
# - vendorItemId / itemId 추출
# - 상품 상세 이미지 URL 추출
# - UI 아이콘 / CSS / JS / 로고 / 썸네일 제외
# - 여러 이미지 / 긴 이미지 1장 모두 지원
# ============================================================

import os
import re
import html as html_lib
from bs4 import BeautifulSoup
from urllib.parse import urlparse


print("=" * 70)
print("🔍 쿠팡 HTML 분석 + 실제 상세 이미지 추출")
print("=" * 70)


# ============================================================
# 1. URL 정규화
# ============================================================

def clean_url(url):

    if not url:
        return ""

    url = html_lib.unescape(str(url))

    # HTML entity
    url = url.replace("&quot;", '"')
    url = url.replace("&#x27;", "'")
    url = url.replace("&#39;", "'")
    url = url.replace("&amp;", "&")

    # 앞뒤 공백
    url = url.strip()

    # HTML 태그 잔여 제거
    url = re.sub(
        r'</?a[^>]*>',
        '',
        url,
        flags=re.I
    )

    url = re.sub(
        r'<[^>]+>',
        '',
        url
    )

    # 따옴표
    url = url.strip('"\' ')

    # protocol-relative
    if url.startswith("//"):
        url = "https:" + url

    return url.strip()


# ============================================================
# 2. 절대 제외 리소스
# ============================================================

def is_bad_url(url):

    if not url:
        return True

    url = clean_url(url)

    lower = url.lower()

    # HTML 찌꺼기
    if "<" in url or ">" in url:
        return True

    if "</a>" in lower:
        return True

    # --------------------------------------------------------
    # 파일 확장자
    # --------------------------------------------------------

    parsed = urlparse(url)

    path = parsed.path.lower()

    bad_extensions = [
        ".css",
        ".js",
        ".json",
        ".xml",
        ".svg",
        ".ico",
        ".txt",
        ".map",
        ".woff",
        ".woff2",
        ".ttf",
        ".otf",
        ".eot",
    ]

    for ext in bad_extensions:

        if path.endswith(ext):
            return True

    # --------------------------------------------------------
    # 정적 리소스 경로
    # --------------------------------------------------------

    bad_paths = [
        "/_next/",
        "/static/",
        "/static/css/",
        "/static/js/",
        "/css/",
        "/js/",
        "/fonts/",
        "/font/",
        "/sprite/",
    ]

    for bad in bad_paths:

        if bad in path:
            return True

    # --------------------------------------------------------
    # 쿠팡 UI 파일명
    # --------------------------------------------------------

    filename = path.split("/")[-1]

    bad_names = [
        "favicon",
        "logo",
        "icon",
        "sprite",
        "arrow",
        "button",
        "loading",
        "spinner",
        "thumbnail",
        "thumb",
        "tiny",
        "badge",
        "rocket-delivery",
        "rocket-fresh",
        "coupang-play",
        "sell-on-coupang",
        "gold-box",
        "oversea-delivery",
        "wow",
        "biz",
        "omp",
    ]

    for name in bad_names:

        if name in filename:
            return True

    return False


# ============================================================
# 3. 이미지 URL인지 검사
# ============================================================

def is_image_url(url):

    url = clean_url(url)

    if not url:
        return False

    if is_bad_url(url):
        return False

    parsed = urlparse(url)

    path = parsed.path.lower()

    # 이미지 확장자
    image_exts = [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".gif",
        ".bmp",
        ".avif",
    ]

    if any(path.endswith(ext) for ext in image_exts):
        return True

    # --------------------------------------------------------
    # Coupang 이미지 서버에서 확장자가 없는 경우
    # --------------------------------------------------------

    image_hints = [
        "/image/retail/images/",
        "/image/",
        "/remote/q",
    ]

    for hint in image_hints:

        if hint in url.lower():
            return True

    return False


# ============================================================
# 4. 중복 제거
# ============================================================

def unique_urls(urls):

    result = []
    seen = set()

    for url in urls:

        url = clean_url(url)

        if not url:
            continue

        if url in seen:
            continue

        if not is_image_url(url):
            continue

        seen.add(url)
        result.append(url)

    return result


# ============================================================
# 5. view-source 복원
# ============================================================

def restore_view_source(raw_html):

    # line-content가 있으면 Chrome view-source
    if "line-content" not in raw_html:
        return raw_html

    soup = BeautifulSoup(
        raw_html,
        "html.parser"
    )

    lines = soup.select(
        "td.line-content"
    )

    if not lines:
        return raw_html

    source = ""

    for line in lines:

        source += line.get_text(
            "",
            strip=False
        )

    return html_lib.unescape(source)


# ============================================================
# 6. vendorItemId
# ============================================================

def get_vendor_item_id(source):

    patterns = [

        r'"vendorItemId"\s*:\s*"?(\d+)',

        r"'vendorItemId'\s*:\s*'?(\d+)",

        r'vendorItemId\s*=\s*["\'](\d+)["\']',

        r'vendorItemId[="\':\s]+(\d+)',
    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            source,
            re.I
        )

        if m:
            return m.group(1)

    return ""


# ============================================================
# 7. itemId
# ============================================================

def get_item_id(source):

    patterns = [

        r'"itemId"\s*:\s*"?(\d+)',

        r"'itemId'\s*:\s*'?(\d+)",

        r'itemId\s*=\s*["\'](\d+)["\']',

        r'itemId[="\':\s]+(\d+)',
    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            source,
            re.I
        )

        if m:
            return m.group(1)

    return ""


# ============================================================
# 8. HTML 속성에서 이미지 URL 추출
# ============================================================

def extract_attribute_images(source):

    urls = []

    # --------------------------------------------------------
    # src / data-src / original 계열
    # --------------------------------------------------------

    patterns = [

        r'(?:src|data-src|data-original|data-lazy-src|data-image)\s*=\s*"([^"]+)"',

        r"(?:src|data-src|data-original|data-lazy-src|data-image)\s*=\s*'([^']+)'",

    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            source,
            flags=re.I
        )

        for url in matches:

            url = clean_url(url)

            if is_image_url(url):
                urls.append(url)

    # --------------------------------------------------------
    # srcset
    # --------------------------------------------------------

    srcset_patterns = [

        r'srcset\s*=\s*"([^"]+)"',

        r"srcset\s*=\s*'([^']+)'",
    ]

    for pattern in srcset_patterns:

        matches = re.findall(
            pattern,
            source,
            flags=re.I
        )

        for srcset in matches:

            for item in srcset.split(","):

                parts = item.strip().split()

                if not parts:
                    continue

                url = clean_url(parts[0])

                if is_image_url(url):
                    urls.append(url)

    return urls


# ============================================================
# 9. JSON / JavaScript 내부 이미지 URL 추출
# ============================================================

def extract_embedded_images(source):

    urls = []

    # --------------------------------------------------------
    # https://... 이미지 URL
    # --------------------------------------------------------

    pattern = (
        r'https?://[^"\'\s<>]+'
    )

    candidates = re.findall(
        pattern,
        source,
        flags=re.I
    )

    for url in candidates:

        url = clean_url(url)

        if is_image_url(url):
            urls.append(url)

    # --------------------------------------------------------
    # //image.coupangcdn.com/... 형태
    # --------------------------------------------------------

    pattern2 = (
        r'//image\.coupangcdn\.com/[^"\'\s<>]+'
    )

    candidates2 = re.findall(
        pattern2,
        source,
        flags=re.I
    )

    for url in candidates2:

        url = "https:" + clean_url(url)

        if is_image_url(url):
            urls.append(url)

    return urls


# ============================================================
# 10. BeautifulSoup에서 상세 이미지 후보 찾기
# ============================================================

def extract_dom_images(source):

    urls = []

    soup = BeautifulSoup(
        source,
        "html.parser"
    )

    # --------------------------------------------------------
    # A. subType-IMAGE
    # --------------------------------------------------------

    containers = soup.select(
        '[class*="subType-IMAGE"]'
    )

    for container in containers:

        for img in container.find_all("img"):

            for attr in [
                "src",
                "data-src",
                "data-original",
                "data-lazy-src",
                "data-image",
            ]:

                value = img.get(attr)

                if not value:
                    continue

                value = clean_url(value)

                if is_image_url(value):
                    urls.append(value)

        # source
        for tag in container.find_all("source"):

            for attr in [
                "src",
                "data-src",
            ]:

                value = tag.get(attr)

                if not value:
                    continue

                value = clean_url(value)

                if is_image_url(value):
                    urls.append(value)

    # --------------------------------------------------------
    # B. 상세영역으로 보이는 img
    # --------------------------------------------------------

    detail_keywords = [
        "product-detail",
        "productDetail",
        "detail-image",
        "detailImage",
        "detail-content",
        "detailContent",
        "vendor-item-detail",
        "item-detail",
    ]

    for keyword in detail_keywords:

        nodes = soup.select(
            f'[class*="{keyword}"]'
        )

        for node in nodes:

            for img in node.find_all("img"):

                for attr in [
                    "src",
                    "data-src",
                    "data-original",
                    "data-lazy-src",
                    "data-image",
                ]:

                    value = img.get(attr)

                    if not value:
                        continue

                    value = clean_url(value)

                    if is_image_url(value):
                        urls.append(value)

    return urls


# ============================================================
# 11. HTML 파일별 처리
# ============================================================

html_records = []


for html_item in html_files:

    # --------------------------------------------------------
    # html_files가 문자열 / dict 모두 처리
    # --------------------------------------------------------

    if isinstance(html_item, dict):

        file_path = (
            html_item.get("path")
            or html_item.get("file_path")
            or html_item.get("filepath")
            or html_item.get("name")
        )

        file_name = (
            html_item.get("name")
            or os.path.basename(str(file_path))
        )

    else:

        file_path = str(html_item)

        file_name = os.path.basename(
            file_path
        )

    print("\n" + "-" * 70)
    print("📄", file_name)

    # --------------------------------------------------------
    # 파일 확인
    # --------------------------------------------------------

    if not file_path or not os.path.exists(file_path):

        print("❌ HTML 파일 없음")

        html_records.append({
            "HTML파일": file_name,
            "vendorItemId": "",
            "itemId": "",
            "상세이미지수": 0,
            "상세이미지URL": [],
            "추출방식": "파일없음",
            "오류": "파일을 찾을 수 없음",
        })

        continue

    # --------------------------------------------------------
    # HTML 읽기
    # --------------------------------------------------------

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            raw_html = f.read()

    except Exception as e:

        print("❌ 읽기 실패:", e)

        html_records.append({
            "HTML파일": file_name,
            "vendorItemId": "",
            "itemId": "",
            "상세이미지수": 0,
            "상세이미지URL": [],
            "추출방식": "읽기실패",
            "오류": str(e),
        })

        continue

    # --------------------------------------------------------
    # view-source 복원
    # --------------------------------------------------------

    source = restore_view_source(
        raw_html
    )

    # --------------------------------------------------------
    # ID
    # --------------------------------------------------------

    vendor_item_id = get_vendor_item_id(
        source
    )

    item_id = get_item_id(
        source
    )

    # --------------------------------------------------------
    # 이미지 추출
    # --------------------------------------------------------

    dom_urls = extract_dom_images(
        source
    )

    # --------------------------------------------------------
    # DOM에서 찾았으면 우선 사용
    # --------------------------------------------------------

    if dom_urls:

        image_urls = unique_urls(
            dom_urls
        )

        method = "DOM 상세영역"

    else:

        # ----------------------------------------------------
        # 속성 전체에서 이미지 후보 추출
        # ----------------------------------------------------

        attr_urls = extract_attribute_images(
            source
        )

        image_urls = unique_urls(
            attr_urls
        )

        if image_urls:
            method = "HTML image attribute"

        else:

            # ------------------------------------------------
            # 최종 embedded URL 검색
            # ------------------------------------------------

            embedded_urls = extract_embedded_images(
                source
            )

            image_urls = unique_urls(
                embedded_urls
            )

            method = "embedded URL"

    # --------------------------------------------------------
    # 결과
    # --------------------------------------------------------

    print(
        "vendorItemId :",
        vendor_item_id
    )

    print(
        "itemId       :",
        item_id
    )

    print(
        "추출방식      :",
        method
    )

    print(
        "실제 상세 이미지 :",
        len(image_urls)
    )

    for i, url in enumerate(
        image_urls[:10],
        1
    ):

        print(
            f"[{i}]",
            url
        )

    if len(image_urls) > 10:

        print(
            f"... 외 {len(image_urls) - 10}개"
        )

    # --------------------------------------------------------
    # 저장
    # --------------------------------------------------------

    html_records.append({
        "HTML파일": file_name,
        "vendorItemId": vendor_item_id,
        "itemId": item_id,
        "상세이미지수": len(image_urls),
        "상세이미지URL": image_urls,
        "추출방식": method,
        "오류": "",
    })


# ============================================================
# 12. 최종 요약
# ============================================================

print("\n" + "=" * 70)
print("✅ HTML 분석 완료")
print("=" * 70)

print(
    "HTML 수 :",
    len(html_records)
)

total_images = sum(
    len(x["상세이미지URL"])
    for x in html_records
)

print(
    "총 상세 이미지 :",
    total_images
)

print("=" * 70)

## **CELL 4 — HTML ↔ Product Master 자동 매칭 + 이미지 다운로드**

In [ ]:
# ============================================================
# CELL 4
# 🔗 HTML ↔ Product Master 매칭
#
# - 현재 Colab에 존재하는 Product Master DataFrame 자동 탐색
# - vendorItemId 우선 매칭
# - itemId 보조 매칭
# - Cell 3의 "상세이미지URL" 사용
# - matched_html 생성
# ============================================================

import pandas as pd


print("=" * 70)
print("🔗 HTML ↔ Product Master 매칭")
print("=" * 70)


# ============================================================
# 1. Product Master DataFrame 자동 찾기
# ============================================================

# 후보 변수명
PM_CANDIDATES = [
    "product_master_df",
    "product_master",
    "df_product_master",
    "pm_df",
    "pm",
    "product_df",
    "df_product",
    "product_master_data",
]


pm = None
pm_var_name = None


# ------------------------------------------------------------
# 후보 변수명에서 찾기
# ------------------------------------------------------------

for var_name in PM_CANDIDATES:

    if var_name in globals():

        obj = globals()[var_name]

        if isinstance(obj, pd.DataFrame):

            pm = obj.copy()
            pm_var_name = var_name
            break


# ------------------------------------------------------------
# 후보 변수명으로 못 찾았으면
# 컬럼 구조를 보고 자동 탐색
# ------------------------------------------------------------

if pm is None:

    dataframe_candidates = []

    for var_name, obj in list(
        globals().items()
    ):

        if not isinstance(
            obj,
            pd.DataFrame
        ):
            continue

        cols = set(
            str(c)
            for c in obj.columns
        )

        score = 0

        if "통합제품ID" in cols:
            score += 5

        if "상품명" in cols:
            score += 3

        if "브랜드" in cols:
            score += 2

        if "상품URL" in cols:
            score += 2

        if "vendorItemId" in cols:
            score += 3

        if "itemId" in cols:
            score += 3

        if score > 0:

            dataframe_candidates.append(
                (
                    score,
                    var_name,
                    obj
                )
            )


    if dataframe_candidates:

        dataframe_candidates.sort(
            key=lambda x: x[0],
            reverse=True
        )

        score, pm_var_name, pm = (
            dataframe_candidates[0]
        )

        pm = pm.copy()


# ------------------------------------------------------------
# Product Master를 찾지 못한 경우
# ------------------------------------------------------------

if pm is None:

    print()
    print("❌ Product Master DataFrame을 찾지 못했습니다.")
    print()
    print("현재 존재하는 DataFrame:")

    for var_name, obj in globals().items():

        if isinstance(
            obj,
            pd.DataFrame
        ):

            print(
                " -",
                var_name,
                obj.shape,
                list(obj.columns)[:10]
            )

    raise NameError(
        "Product Master DataFrame을 찾을 수 없습니다."
    )


print(
    "Product Master 변수 :",
    pm_var_name
)

print(
    "Product Master 행 수 :",
    len(pm)
)


# ============================================================
# 2. Product Master 필수 컬럼 확인
# ============================================================

required_cols = [
    "통합제품ID",
    "브랜드",
    "상품명",
]


for col in required_cols:

    if col not in pm.columns:

        pm[col] = ""


if "itemId" not in pm.columns:

    pm["itemId"] = ""


if "vendorItemId" not in pm.columns:

    pm["vendorItemId"] = ""


if "상품URL" not in pm.columns:

    pm["상품URL"] = ""


# ============================================================
# 3. ID 문자열 정리
# ============================================================

for col in [
    "통합제품ID",
    "브랜드",
    "상품명",
    "상품URL",
    "itemId",
    "vendorItemId",
]:

    pm[col] = (
        pm[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


# ============================================================
# 4. Product Master 매칭 Dictionary
# ============================================================

vendor_map = {}
item_map = {}


# ------------------------------------------------------------
# vendorItemId
# ------------------------------------------------------------

for _, row in pm.iterrows():

    vendor_id = str(
        row["vendorItemId"]
    ).strip()

    if vendor_id:

        vendor_map.setdefault(
            vendor_id,
            row
        )


# ------------------------------------------------------------
# itemId
# ------------------------------------------------------------

for _, row in pm.iterrows():

    item_id = str(
        row["itemId"]
    ).strip()

    if item_id:

        item_map.setdefault(
            item_id,
            row
        )


# ============================================================
# 5. HTML ↔ Product Master 매칭
# ============================================================

matched_html = []
failed_html = []


for html_record in html_records:

    vendor_id = str(
        html_record.get(
            "vendorItemId",
            ""
        )
    ).strip()

    item_id = str(
        html_record.get(
            "itemId",
            ""
        )
    ).strip()


    # --------------------------------------------------------
    # vendorItemId 우선
    # --------------------------------------------------------

    master_row = None
    match_type = ""


    if (
        vendor_id
        and vendor_id in vendor_map
    ):

        master_row = vendor_map[
            vendor_id
        ]

        match_type = "vendorItemId"


    # --------------------------------------------------------
    # itemId 보조
    # --------------------------------------------------------

    elif (
        item_id
        and item_id in item_map
    ):

        master_row = item_map[
            item_id
        ]

        match_type = "itemId"


    # --------------------------------------------------------
    # 실패
    # --------------------------------------------------------

    if master_row is None:

        failed_html.append({

            "HTML파일":
                html_record.get(
                    "HTML파일",
                    ""
                ),

            "vendorItemId":
                vendor_id,

            "itemId":
                item_id,

            "상세이미지수":
                html_record.get(
                    "상세이미지수",
                    0
                ),

            "오류":
                "Product Master 매칭 실패",
        })

        continue


    # ========================================================
    # 상세 이미지 URL
    #
    # Cell 3에서 생성한 정확한 컬럼
    # ========================================================

    image_urls = html_record.get(
        "상세이미지URL",
        []
    )


    if image_urls is None:

        image_urls = []


    # 문자열이면 리스트화
    if isinstance(
        image_urls,
        str
    ):

        image_urls = (
            [image_urls]
            if image_urls.strip()
            else []
        )


    # ========================================================
    # Product Master URL
    # ========================================================

    master_url = ""

    try:

        master_url = str(
            master_row.get(
                "상품URL",
                ""
            )
        ).strip()

    except Exception:
        master_url = ""


    # ========================================================
    # matched_html
    # ========================================================

    matched_html.append({

        "통합제품ID":
            str(
                master_row["통합제품ID"]
            ).strip(),

        "상품URL":
            master_url,

        "브랜드":
            str(
                master_row["브랜드"]
            ).strip(),

        "상품명":
            str(
                master_row["상품명"]
            ).strip(),

        "itemId":
            item_id,

        "vendorItemId":
            vendor_id,

        "HTML파일":
            html_record.get(
                "HTML파일",
                ""
            ),

        # ★ Cell 5에서 사용할 이미지 리스트
        "이미지URL":
            image_urls,

        "상세이미지수":
            len(image_urls),

        "매칭기준":
            match_type,

        "추출방식":
            html_record.get(
                "추출방식",
                ""
            ),

        "오류":
            html_record.get(
                "오류",
                ""
            ),
    })


# ============================================================
# 6. DataFrame 생성
# ============================================================

MATCHED_COLUMNS = [
    "통합제품ID",
    "상품URL",
    "브랜드",
    "상품명",
    "itemId",
    "vendorItemId",
    "HTML파일",
    "이미지URL",
    "상세이미지수",
    "매칭기준",
    "추출방식",
    "오류",
]


matched_html_df = pd.DataFrame(
    matched_html,
    columns=MATCHED_COLUMNS
)


FAILED_COLUMNS = [
    "HTML파일",
    "vendorItemId",
    "itemId",
    "상세이미지수",
    "오류",
]


failed_html_df = pd.DataFrame(
    failed_html,
    columns=FAILED_COLUMNS
)


# ============================================================
# 7. 매칭 결과
# ============================================================

print()
print("=" * 70)
print("🔗 매칭 결과")
print("=" * 70)

print(
    "Product Master :",
    len(pm)
)

print(
    "HTML           :",
    len(html_records)
)

print(
    "매칭 성공       :",
    len(matched_html_df)
)

print(
    "매칭 실패       :",
    len(failed_html_df)
)


# ============================================================
# 8. 성공 상품 확인
# ============================================================

if len(matched_html_df) > 0:

    print()
    print("-" * 70)
    print("📦 매칭 성공 상품")
    print("-" * 70)

    for _, row in matched_html_df.iterrows():

        print(
            "통합제품ID :",
            row["통합제품ID"]
        )

        print(
            "브랜드      :",
            row["브랜드"]
        )

        print(
            "상품명      :",
            row["상품명"]
        )

        print(
            "vendorItemId :",
            row["vendorItemId"]
        )

        print(
            "itemId       :",
            row["itemId"]
        )

        print(
            "매칭기준      :",
            row["매칭기준"]
        )

        print(
            "상세이미지수  :",
            row["상세이미지수"]
        )

        print(
            "추출방식      :",
            row["추출방식"]
        )

        print()


# ============================================================
# 9. 이미지 URL 확인
# ============================================================

if len(matched_html_df) > 0:

    print("-" * 70)
    print("🖼️ 이미지 URL 샘플")
    print("-" * 70)

    for _, row in matched_html_df.iterrows():

        urls = row["이미지URL"]

        for i, url in enumerate(
            urls[:10],
            1
        ):

            print(
                f"[{i}]",
                url
            )

        if len(urls) > 10:

            print(
                f"... 외 {len(urls) - 10}개"
            )


# ============================================================
# 10. 실패 HTML
# ============================================================

if len(failed_html_df) > 0:

    print()
    print("-" * 70)
    print("❌ 매칭 실패 HTML")
    print("-" * 70)

    display(
        failed_html_df
    )

else:

    print(
        "❌ 매칭 실패 HTML 없음"
    )


print("=" * 70)

## **CELL 5 — OCR + 상품 정보 추출**

In [ ]:
# ============================================================
# CELL 5
# 🖼️ 상세 이미지 다운로드 + 이미지별 OCR
#
# 입력:
#   matched_html_df
#
# 출력:
#   image_download_records
#   ocr_detail_records
#   ocr_product_records
#
# 핵심:
#   - Cell 4의 이미지URL을 그대로 사용
#   - http / https 모두 허용
#   - 실제 이미지 응답인지 확인
#   - 이미지별로 개별 다운로드
#   - 이미지별로 개별 OCR
#   - OCR 전문은 OCR전문용 데이터로 보존
#   - 이미지를 서로 합치지 않음
# ============================================================

import os
import re
import io
import time
import requests
import pandas as pd

from PIL import Image, ImageOps, ImageEnhance
import pytesseract


# ============================================================
# 0. 경로
# ============================================================

BASE_DIR = "/content/coupang_ocr"

HTML_DIR = os.path.join(
    BASE_DIR,
    "html"
)

IMAGE_DIR = os.path.join(
    BASE_DIR,
    "images"
)

RESULT_DIR = os.path.join(
    BASE_DIR,
    "results"
)

os.makedirs(
    IMAGE_DIR,
    exist_ok=True
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 1. OCR 환경 확인
# ============================================================

print("=" * 70)
print("🖼️ 상세 이미지 필터링 + 다운로드 + OCR 시작")
print("=" * 70)


try:

    langs = pytesseract.get_languages(
        config=""
    )

    if "kor" in langs and "eng" in langs:

        OCR_LANG = "kor+eng"

    elif "kor" in langs:

        OCR_LANG = "kor"

    elif "eng" in langs:

        OCR_LANG = "eng"

    else:

        OCR_LANG = "eng"


    print(
        "✅ OCR 환경 확인 완료"
    )

    print(
        "OCR 언어 :",
        OCR_LANG
    )

except Exception as e:

    print(
        "⚠️ OCR 환경 확인 실패:",
        e
    )

    OCR_LANG = "eng"


# ============================================================
# 2. HTTP Session
# ============================================================

session = requests.Session()

session.headers.update({

    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36",

    "Accept":
        "image/avif,image/webp,image/apng,"
        "image/svg+xml,image/*,*/*;q=0.8",

    "Accept-Language":
        "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",

    "Referer":
        "https://www.coupang.com/",
})


# ============================================================
# 3. URL 정리
# ============================================================

def clean_image_url(url):

    if url is None:
        return ""

    url = str(url).strip()

    if not url:
        return ""

    # HTML entity
    url = (
        url
        .replace("&amp;", "&")
        .replace("&quot;", '"')
        .replace("&#39;", "'")
    )

    # 앞뒤 따옴표 제거
    url = url.strip(
        "\"'"
    )

    # 혹시 남아있는 HTML 태그 제거
    url = re.sub(
        r"<[^>]+>",
        "",
        url
    ).strip()

    return url


# ============================================================
# 4. 명백한 비이미지 URL 제외
#
# 주의:
#   coupangcdn.com 자체를 제외하지 않는다.
#   현재 실제 상세 이미지는 coupangcdn.com이다.
# ============================================================

def is_obviously_bad_url(url):

    if not url:
        return True

    lower = url.lower()

    # 파일/리소스 확장자
    bad_extensions = [
        ".css",
        ".js",
        ".json",
        ".xml",
        ".woff",
        ".woff2",
        ".ttf",
        ".otf",
        ".map",
    ]

    for ext in bad_extensions:

        if ext in lower:
            return True


    # 정적 프론트 리소스
    bad_patterns = [

        "/_next/",
        "/static/css/",
        "/static/js/",
        "/static/chunks/",
        "javascript:",
        "data:text/",
        "blob:",
    ]

    for pattern in bad_patterns:

        if pattern in lower:
            return True


    return False


# ============================================================
# 5. 이미지 URL 정리
#
# Cell 3에서 이미 상세영역으로 추출했기 때문에
# 과도한 필터링을 하지 않는다.
# ============================================================

def normalize_image_urls(urls):

    if urls is None:

        return []


    if isinstance(
        urls,
        str
    ):

        urls = [
            urls
        ]


    result = []
    seen = set()


    for url in urls:

        url = clean_image_url(
            url
        )

        if not url:
            continue

        if is_obviously_bad_url(
            url
        ):
            continue

        # http / https 모두 허용
        if not url.startswith(
            (
                "http://",
                "https://"
            )
        ):
            continue


        if url in seen:
            continue

        seen.add(url)

        result.append(url)


    return result


# ============================================================
# 6. OCR 전처리
# ============================================================

def prepare_image_for_ocr(img):

    # RGB
    if img.mode != "RGB":

        img = img.convert(
            "RGB"
        )


    # 너무 작은 이미지는 확대
    width, height = img.size


    if width < 1200:

        scale = 2

        img = img.resize(
            (
                width * scale,
                height * scale
            ),
            Image.Resampling.LANCZOS
        )


    # grayscale
    gray = ImageOps.grayscale(
        img
    )


    # 대비 강화
    gray = ImageEnhance.Contrast(
        gray
    ).enhance(1.5)


    return gray


# ============================================================
# 7. OCR 실행
# ============================================================

def run_ocr(img):

    try:

        prepared = prepare_image_for_ocr(
            img
        )

        text = pytesseract.image_to_string(
            prepared,
            lang=OCR_LANG,
            config="--psm 6"
        )


        if text is None:

            return ""


        return str(text).strip()


    except Exception:

        return ""


# ============================================================
# 8. OCR 결과에서 기본 정보 추출
# ============================================================

def extract_volume(text):

    if not text:
        return ""


    patterns = [

        # ml / mL
        r"(\d+(?:\.\d+)?)\s*(?:mL|ml|ML|㎖)",

        # g
        r"(\d+(?:\.\d+)?)\s*(?:g|G|그램)",

        # kg
        r"(\d+(?:\.\d+)?)\s*(?:kg|KG|킬로그램)",

        # L
        r"(\d+(?:\.\d+)?)\s*(?:L|ℓ)",

        # oz
        r"(\d+(?:\.\d+)?)\s*(?:oz|OZ)",
    ]


    for pattern in patterns:

        m = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if m:

            return m.group(0).strip()


    return ""


# ============================================================
# 9. 개수 추출
# ============================================================

def extract_count(text):

    if not text:
        return ""


    patterns = [

        r"(\d+)\s*(?:개입|개|입)",

        r"(?:총|약)?\s*(\d+)\s*(?:개입|개)",

        r"(\d+)\s*(?:ea|EA|pcs|PCS)",

        r"(\d+)\s*팩",

        r"(\d+)\s*봉",

        r"(\d+)\s*캔",

        r"(\d+)\s*병",

    ]


    for pattern in patterns:

        m = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if m:

            return m.group(0).strip()


    return ""


# ============================================================
# 10. 향 / 맛 추출
# ============================================================

def extract_scent(text):

    if not text:
        return ""


    # 향/맛/플레이버 등이 명시된 경우
    patterns = [

        r"(?:향|맛|플레이버)\s*[:：]?\s*([^\n,;/]+)",

        r"([가-힣A-Za-z]+(?:맛|향))",

    ]


    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            re.IGNORECASE
        )

        if matches:

            value = str(
                matches[0]
            ).strip()

            if value:

                return value


    return ""


# ============================================================
# 11. OCR 제품명 후보
#
# 너무 공격적으로 추정하지 않고
# OCR 텍스트의 첫 번째 의미 있는 줄만 사용
# ============================================================

def extract_product_name(text):

    if not text:
        return ""


    lines = []

    for line in text.splitlines():

        line = re.sub(
            r"\s+",
            " ",
            line
        ).strip()

        if not line:
            continue

        if len(line) < 2:
            continue

        lines.append(
            line
        )


    if not lines:
        return ""


    # 너무 긴 영양성분/주의문구 등은 제외
    bad_words = [
        "영양정보",
        "영양성분",
        "주의사항",
        "보관방법",
        "식품유형",
        "원재료명",
        "고객센터",
        "www.",
        "http",
    ]


    for line in lines:

        if any(
            word in line
            for word in bad_words
        ):
            continue

        # 숫자만 있는 줄 제외
        if re.fullmatch(
            r"[\d\s.,:/()%+-]+",
            line
        ):
            continue

        return line


    return lines[0]


# ============================================================
# 12. OCR 브랜드 후보
#
# 명확한 브랜드 키워드가 있는 경우만 추출
# ============================================================

KNOWN_BRANDS = [

    "자유시간",
    "트윅스",
    "스니커즈",
    "맥스봉",
    "기린",
    "미닛메이드",
    "갈아만든배",
    "오리온",
    "롯데",
    "해태",
    "농심",
    "크라운",
    "동서",
    "삼립",
    "초코파이",
    "빼빼로",
]


def extract_brand(text):

    if not text:
        return ""


    found = []


    for brand in KNOWN_BRANDS:

        if brand.lower() in text.lower():

            found.append(
                brand
            )


    # 중복 제거
    found = list(
        dict.fromkeys(
            found
        )
    )


    if found:

        return ", ".join(
            found
        )


    return ""


# ============================================================
# 13. 기록용 리스트
# ============================================================

image_download_records = []

ocr_detail_records = []

ocr_product_records = []


# ============================================================
# 14. 전체 상품 처리
# ============================================================

total_products = len(
    matched_html_df
)


for product_idx, row in matched_html_df.iterrows():

    product_id = str(
        row.get(
            "통합제품ID",
            ""
        )
    ).strip()

    product_url = str(
        row.get(
            "상품URL",
            ""
        )
    ).strip()

    master_brand = str(
        row.get(
            "브랜드",
            ""
        )
    ).strip()

    master_product_name = str(
        row.get(
            "상품명",
            ""
        )
    ).strip()

    vendor_item_id = str(
        row.get(
            "vendorItemId",
            ""
        )
    ).strip()

    item_id = str(
        row.get(
            "itemId",
            ""
        )
    ).strip()

    html_file = str(
        row.get(
            "HTML파일",
            ""
        )
    ).strip()


    # ========================================================
    # 이미지 URL
    # ========================================================

    raw_urls = row.get(
        "이미지URL",
        []
    )

    image_urls = normalize_image_urls(
        raw_urls
    )


    print()
    print("=" * 70)

    print(
        f"[{product_idx + 1}/{total_products}] "
        f"{product_id} | "
        f"{master_product_name}"
    )

    print(
        "HTML 이미지 URL :",
        len(raw_urls)
        if isinstance(raw_urls, list)
        else 0
    )

    print(
        "상세 이미지 후보 :",
        len(image_urls)
    )


    # ========================================================
    # 이미지 없음
    # ========================================================

    if len(image_urls) == 0:

        ocr_product_records.append({

            "통합제품ID":
                product_id,

            "상품URL":
                product_url,

            "브랜드":
                master_brand,

            "상품명":
                master_product_name,

            "OCR브랜드":
                "",

            "OCR제품명":
                "",

            "OCR용량":
                "",

            "OCR향":
                "",

            "OCR개수":
                "",

            "상세이미지수":
                0,

            "다운로드이미지수":
                0,

            "OCR성공이미지수":
                0,

            "OCR텍스트":
                "",

            "다운로드실패수":
                0,

            "처리상태":
                "이미지없음",

            "오류":
                "상세 이미지 URL 없음",

            "vendorItemId":
                vendor_item_id,

            "itemId":
                item_id,

            "HTML파일":
                html_file,
        })

        print(
            "  → 상세 이미지 : 0"
        )

        print(
            "  → 다운로드 성공 : 0"
        )

        print(
            "  → 다운로드 실패 : 0"
        )

        print(
            "  → OCR 성공 : 0"
        )

        continue


    # ========================================================
    # 상품별 이미지 디렉토리
    # ========================================================

    product_image_dir = os.path.join(
        IMAGE_DIR,
        product_id
    )

    os.makedirs(
        product_image_dir,
        exist_ok=True
    )


    # ========================================================
    # 상품별 OCR 텍스트
    # ========================================================

    product_ocr_parts = []

    product_ocr_brands = []
    product_ocr_names = []
    product_ocr_volumes = []
    product_ocr_scents = []
    product_ocr_counts = []


    download_success_count = 0
    download_fail_count = 0
    ocr_success_count = 0


    product_errors = []


    # ========================================================
    # 이미지별 처리
    # ========================================================

    for image_no, image_url in enumerate(
        image_urls,
        start=1
    ):

        ext = ".jpg"

        filename = (
            f"img_{image_no:03d}"
            f"{ext}"
        )

        save_path = os.path.join(
            product_image_dir,
            filename
        )


        download_status = "실패"
        http_status = ""
        download_error = ""

        image_width = ""
        image_height = ""

        ocr_success = False
        ocr_text = ""
        ocr_error = ""


        # ====================================================
        # 다운로드
        # ====================================================

        try:

            response = session.get(
                image_url,
                timeout=30
            )

            http_status = response.status_code


            if response.status_code == 200:

                content_type = (
                    response.headers
                    .get(
                        "Content-Type",
                        ""
                    )
                    .lower()
                )


                # 실제 이미지인지 PIL로 확인
                img = Image.open(
                    io.BytesIO(
                        response.content
                    )
                )

                img.load()

                image_width, image_height = (
                    img.size
                )


                # 확장자 결정
                fmt = (
                    img.format
                    or ""
                ).upper()


                if fmt == "PNG":

                    filename = (
                        f"img_{image_no:03d}.png"
                    )

                elif fmt == "WEBP":

                    filename = (
                        f"img_{image_no:03d}.webp"
                    )

                elif fmt == "JPEG":

                    filename = (
                        f"img_{image_no:03d}.jpg"
                    )

                else:

                    filename = (
                        f"img_{image_no:03d}.jpg"
                    )


                save_path = os.path.join(
                    product_image_dir,
                    filename
                )


                # 원본 저장
                with open(
                    save_path,
                    "wb"
                ) as f:

                    f.write(
                        response.content
                    )


                download_status = "성공"
                download_success_count += 1


                # =================================================
                # OCR
                # =================================================

                try:

                    ocr_text = run_ocr(
                        img
                    )


                    if ocr_text.strip():

                        ocr_success = True
                        ocr_success_count += 1


                        # 제품별 OCR 합치기
                        product_ocr_parts.append(
                            f"[이미지 {image_no}]\n"
                            f"{ocr_text.strip()}"
                        )


                        # OCR 항목
                        brand_value = (
                            extract_brand(
                                ocr_text
                            )
                        )

                        name_value = (
                            extract_product_name(
                                ocr_text
                            )
                        )

                        volume_value = (
                            extract_volume(
                                ocr_text
                            )
                        )

                        scent_value = (
                            extract_scent(
                                ocr_text
                            )
                        )

                        count_value = (
                            extract_count(
                                ocr_text
                            )
                        )


                        if brand_value:

                            product_ocr_brands.append(
                                brand_value
                            )


                        if name_value:

                            product_ocr_names.append(
                                name_value
                            )


                        if volume_value:

                            product_ocr_volumes.append(
                                volume_value
                            )


                        if scent_value:

                            product_ocr_scents.append(
                                scent_value
                            )


                        if count_value:

                            product_ocr_counts.append(
                                count_value
                            )


                    else:

                        ocr_error = (
                            "OCR 결과 없음"
                        )


                except Exception as e:

                    ocr_error = (
                        f"OCR 오류: {e}"
                    )


            else:

                download_error = (
                    f"HTTP {response.status_code}"
                )

                download_fail_count += 1


        except Exception as e:

            download_error = str(e)

            download_fail_count += 1


        # ====================================================
        # 이미지 다운로드 기록
        # ====================================================

        image_download_records.append({

            "통합제품ID":
                product_id,

            "상품URL":
                product_url,

            "vendorItemId":
                vendor_item_id,

            "itemId":
                item_id,

            "이미지번호":
                image_no,

            "이미지URL":
                image_url,

            "이미지파일":
                filename,

            "저장경로":
                save_path
                if download_status == "성공"
                else "",

            "다운로드상태":
                download_status,

            "HTTP상태":
                http_status,

            "이미지너비":
                image_width,

            "이미지높이":
                image_height,

            "오류":
                download_error,
        })


        # ====================================================
        # OCR 전문
        #
        # ★ 다운로드 실패한 이미지도 한 줄 남긴다.
        # ====================================================

        ocr_detail_records.append({

            "통합제품ID":
                product_id,

            "상품URL":
                product_url,

            "브랜드":
                master_brand,

            "상품명":
                master_product_name,

            "vendorItemId":
                vendor_item_id,

            "itemId":
                item_id,

            "HTML파일":
                html_file,

            "이미지번호":
                image_no,

            "이미지URL":
                image_url,

            "이미지파일":
                filename,

            "다운로드상태":
                download_status,

            "HTTP상태":
                http_status,

            "이미지너비":
                image_width,

            "이미지높이":
                image_height,

            "OCR성공":
                "성공"
                if ocr_success
                else "실패",

            "OCR전문":
                ocr_text,

            "오류":
                ocr_error
                or download_error,
        })


        # 진행 표시
        print(
            f"  [{image_no:02d}/{len(image_urls):02d}] "
            f"{download_status} | "
            f"OCR "
            f"{'성공' if ocr_success else '실패'}"
        )


    # ========================================================
    # 중복 제거
    # ========================================================

    def unique_join(values):

        result = []

        seen = set()

        for value in values:

            value = str(
                value
            ).strip()

            if not value:
                continue

            if value in seen:
                continue

            seen.add(value)

            result.append(
                value
            )

        return " / ".join(
            result
        )


    # ========================================================
    # 상품별 OCR 텍스트
    # ========================================================

    combined_ocr_text = (
        "\n\n".join(
            product_ocr_parts
        )
    )


    # ========================================================
    # 상품 결과
    # ========================================================

    if (
        download_fail_count == 0
        and ocr_success_count == len(image_urls)
    ):

        status = "완료"

    elif download_success_count > 0:

        status = "부분완료"

    else:

        status = "다운로드실패"


    ocr_product_records.append({

        "통합제품ID":
            product_id,

        "상품URL":
            product_url,

        "브랜드":
            master_brand,

        "상품명":
            master_product_name,

        "OCR브랜드":
            unique_join(
                product_ocr_brands
            ),

        "OCR제품명":
            unique_join(
                product_ocr_names
            ),

        "OCR용량":
            unique_join(
                product_ocr_volumes
            ),

        "OCR향":
            unique_join(
                product_ocr_scents
            ),

        "OCR개수":
            unique_join(
                product_ocr_counts
            ),

        "상세이미지수":
            len(image_urls),

        "다운로드이미지수":
            download_success_count,

        "OCR성공이미지수":
            ocr_success_count,

        "OCR텍스트":
            combined_ocr_text,

        "다운로드실패수":
            download_fail_count,

        "처리상태":
            status,

        "오류":
            " / ".join(
                product_errors
            ),

        "vendorItemId":
            vendor_item_id,

        "itemId":
            item_id,

        "HTML파일":
            html_file,
    })


    # ========================================================
    # 상품 결과 출력
    # ========================================================

    print(
        f"  → 상세 이미지 : {len(image_urls)}"
    )

    print(
        f"  → 다운로드 성공 : {download_success_count}"
    )

    print(
        f"  → 다운로드 실패 : {download_fail_count}"
    )

    print(
        f"  → OCR 성공 : {ocr_success_count}"
    )


# ============================================================
# 15. DataFrame 생성
# ============================================================

OCR_RESULT_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "브랜드",
    "상품명",
    "OCR브랜드",
    "OCR제품명",
    "OCR용량",
    "OCR향",
    "OCR개수",
    "상세이미지수",
    "다운로드이미지수",
    "OCR성공이미지수",
    "OCR텍스트",
    "다운로드실패수",
    "처리상태",
    "오류",
    "vendorItemId",
    "itemId",
    "HTML파일",
]


OCR_DETAIL_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "브랜드",
    "상품명",
    "vendorItemId",
    "itemId",
    "HTML파일",
    "이미지번호",
    "이미지URL",
    "이미지파일",
    "다운로드상태",
    "HTTP상태",
    "이미지너비",
    "이미지높이",
    "OCR성공",
    "OCR전문",
    "오류",
]


IMAGE_DOWNLOAD_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "vendorItemId",
    "itemId",
    "이미지번호",
    "이미지URL",
    "이미지파일",
    "저장경로",
    "다운로드상태",
    "HTTP상태",
    "이미지너비",
    "이미지높이",
    "오류",
]


ocr_results_df = pd.DataFrame(
    ocr_product_records,
    columns=OCR_RESULT_COLUMNS
)


ocr_detail_df = pd.DataFrame(
    ocr_detail_records,
    columns=OCR_DETAIL_COLUMNS
)


image_download_df = pd.DataFrame(
    image_download_records,
    columns=IMAGE_DOWNLOAD_COLUMNS
)


# ============================================================
# 16. 전역 변수
# ============================================================

# 다음 Cell에서 그대로 사용
ocr_results = ocr_results_df
ocr_detail_results = ocr_detail_df
image_download_results = image_download_df


# ============================================================
# 17. 최종 요약
# ============================================================

print()
print("=" * 70)
print("✅ 이미지 다운로드 + OCR 완료")
print("=" * 70)

print(
    "상품 결과 :",
    len(ocr_results_df)
)

print(
    "이미지별 결과 :",
    len(ocr_detail_df)
)

print(
    "다운로드 기록 :",
    len(image_download_df)
)


if len(ocr_detail_df) > 0:

    print()
    print(
        "📊 이미지 처리 현황"
    )

    print(
        "다운로드 성공 :",
        (
            image_download_df[
                "다운로드상태"
            ] == "성공"
        ).sum()
    )

    print(
        "다운로드 실패 :",
        (
            image_download_df[
                "다운로드상태"
            ] == "실패"
        ).sum()
    )

    print(
        "OCR 성공 :",
        (
            ocr_detail_df[
                "OCR성공"
            ] == "성공"
        ).sum()
    )


print("=" * 70)

## **CELL 6 — 최종 Excel 저장**

In [ ]:
# ============================================================
# CELL 6
# 📊 최종 Excel 저장
#
# 현재 Cell 3~5에서 생성된 DataFrame을 그대로 사용
#
# Sheet
#   1. OCR결과
#   2. OCR전문
#   3. ProductMaster
#   4. HTML분석
#   5. 이미지다운로드
#   6. HTML매칭실패
# ============================================================

import os
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


print("=" * 70)
print("📊 최종 Excel 저장")
print("=" * 70)


# ============================================================
# 1. 저장 경로
# ============================================================

BASE_DIR = "/content/coupang_ocr"

RESULT_DIR = os.path.join(
    BASE_DIR,
    "results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


OUTPUT_FILE = os.path.join(
    RESULT_DIR,
    "coupang_ocr_raw.xlsx"
)


# ============================================================
# 2. DataFrame 안전 처리
# ============================================================

def safe_df(
    df,
    columns
):

    # DataFrame이 아니면 빈 DataFrame
    if not isinstance(
        df,
        pd.DataFrame
    ):

        return pd.DataFrame(
            columns=columns
        )


    # 필요한 컬럼이 없으면 생성
    for col in columns:

        if col not in df.columns:

            df[col] = ""


    # 지정 순서
    return df[
        columns
    ].copy()


# ============================================================
# 3. OCR결과
# ============================================================

OCR_RESULT_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "브랜드",
    "상품명",
    "OCR브랜드",
    "OCR제품명",
    "OCR용량",
    "OCR향",
    "OCR개수",
    "상세이미지수",
    "다운로드이미지수",
    "OCR성공이미지수",
    "OCR텍스트",
    "다운로드실패수",
    "처리상태",
    "오류",
    "vendorItemId",
    "itemId",
    "HTML파일",

]


ocr_results_export = safe_df(
    ocr_results_df,
    OCR_RESULT_COLUMNS
)


# ============================================================
# 4. OCR전문
# ============================================================

OCR_DETAIL_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "브랜드",
    "상품명",
    "vendorItemId",
    "itemId",
    "HTML파일",
    "이미지번호",
    "이미지URL",
    "이미지파일",
    "다운로드상태",
    "HTTP상태",
    "이미지너비",
    "이미지높이",
    "OCR성공",
    "OCR전문",
    "오류",

]


ocr_detail_export = safe_df(
    ocr_detail_df,
    OCR_DETAIL_COLUMNS
)


# ============================================================
# 5. ProductMaster
# ============================================================

PRODUCT_MASTER_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "브랜드",
    "상품명",
    "itemId",
    "vendorItemId",

]


product_master_export = safe_df(
    pm,
    PRODUCT_MASTER_COLUMNS
)


# ============================================================
# 6. HTML분석
# ============================================================

HTML_ANALYSIS_COLUMNS = [

    "HTML파일",
    "vendorItemId",
    "itemId",
    "상세이미지수",
    "상세이미지URL",
    "추출방식",
    "오류",

]


html_analysis_export = safe_df(
    html_analysis_df,
    HTML_ANALYSIS_COLUMNS
)


# ============================================================
# 7. 이미지다운로드
# ============================================================

IMAGE_DOWNLOAD_COLUMNS = [

    "통합제품ID",
    "상품URL",
    "vendorItemId",
    "itemId",
    "이미지번호",
    "이미지URL",
    "이미지파일",
    "저장경로",
    "다운로드상태",
    "HTTP상태",
    "이미지너비",
    "이미지높이",
    "오류",

]


image_download_export = safe_df(
    image_download_df,
    IMAGE_DOWNLOAD_COLUMNS
)


# ============================================================
# 8. HTML매칭실패
# ============================================================

HTML_FAILED_COLUMNS = [

    "HTML파일",
    "vendorItemId",
    "itemId",
    "상세이미지수",
    "오류",

]


failed_html_export = safe_df(
    failed_html_df,
    HTML_FAILED_COLUMNS
)


# ============================================================
# 9. Excel 저장
# ============================================================

print()
print("💾 Excel 저장 중...")


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # OCR결과
    # --------------------------------------------------------

    ocr_results_export.to_excel(
        writer,
        sheet_name="OCR결과",
        index=False
    )


    # --------------------------------------------------------
    # OCR전문
    # --------------------------------------------------------

    ocr_detail_export.to_excel(
        writer,
        sheet_name="OCR전문",
        index=False
    )


    # --------------------------------------------------------
    # ProductMaster
    # --------------------------------------------------------

    product_master_export.to_excel(
        writer,
        sheet_name="ProductMaster",
        index=False
    )


    # --------------------------------------------------------
    # HTML분석
    # --------------------------------------------------------

    html_analysis_export.to_excel(
        writer,
        sheet_name="HTML분석",
        index=False
    )


    # --------------------------------------------------------
    # 이미지다운로드
    # --------------------------------------------------------

    image_download_export.to_excel(
        writer,
        sheet_name="이미지다운로드",
        index=False
    )


    # --------------------------------------------------------
    # HTML매칭실패
    # --------------------------------------------------------

    failed_html_export.to_excel(
        writer,
        sheet_name="HTML매칭실패",
        index=False
    )


# ============================================================
# 10. Excel 서식
# ============================================================

print(
    "🎨 Excel 서식 적용 중..."
)


wb = load_workbook(
    OUTPUT_FILE
)


for ws in wb.worksheets:

    # --------------------------------------------------------
    # 첫 번째 행
    # --------------------------------------------------------

    if ws.max_row >= 1:

        for cell in ws[1]:

            cell.font = Font(
                bold=True
            )

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )


    # --------------------------------------------------------
    # 틀 고정
    # --------------------------------------------------------

    ws.freeze_panes = "A2"


    # --------------------------------------------------------
    # 자동 너비
    # --------------------------------------------------------

    for column_cells in ws.columns:

        max_length = 0

        column_letter = (
            get_column_letter(
                column_cells[0].column
            )
        )


        for cell in column_cells:

            try:

                value = str(
                    cell.value
                )

                # 너무 긴 OCR 전문은
                # 열 너비 계산에서 제한
                length = min(
                    len(value),
                    50
                )

                if length > max_length:

                    max_length = length

            except Exception:

                pass


        width = min(
            max(
                max_length + 2,
                10
            ),
            60
        )


        ws.column_dimensions[
            column_letter
        ].width = width


    # --------------------------------------------------------
    # OCR 전문 / OCR 텍스트 열
    # --------------------------------------------------------

    for cell in ws[1]:

        if cell.value in [
            "OCR전문",
            "OCR텍스트",
            "상세이미지URL",
        ]:

            col_letter = (
                get_column_letter(
                    cell.column
                )
            )

            ws.column_dimensions[
                col_letter
            ].width = 80


# ============================================================
# 11. 저장
# ============================================================

wb.save(
    OUTPUT_FILE
)


# ============================================================
# 12. 결과 확인
# ============================================================

print()
print("=" * 70)
print("✅ 최종 Excel 저장 완료")
print("=" * 70)

print(
    "파일 :",
    OUTPUT_FILE
)

print()


# ============================================================
# Sheet별 행 수
# ============================================================

print("📊 Sheet별 데이터")

print(
    "OCR결과       :",
    len(ocr_results_export),
    "행"
)

print(
    "OCR전문       :",
    len(ocr_detail_export),
    "행"
)

print(
    "ProductMaster :",
    len(product_master_export),
    "행"
)

print(
    "HTML분석      :",
    len(html_analysis_export),
    "행"
)

print(
    "이미지다운로드 :",
    len(image_download_export),
    "행"
)

print(
    "HTML매칭실패   :",
    len(failed_html_export),
    "행"
)


# ============================================================
# 13. 현재 상품 결과
# ============================================================

if len(ocr_results_export) > 0:

    print()
    print("-" * 70)
    print("📦 OCR 상품 결과")
    print("-" * 70)


    display(
        ocr_results_export[
            [
                "통합제품ID",
                "브랜드",
                "상품명",
                "상세이미지수",
                "다운로드이미지수",
                "OCR성공이미지수",
                "다운로드실패수",
                "처리상태",
            ]
        ]
    )


# ============================================================
# 14. OCR전문 미리보기
# ============================================================

if len(ocr_detail_export) > 0:

    print()
    print("-" * 70)
    print("🔍 OCR전문 미리보기")
    print("-" * 70)


    display(
        ocr_detail_export[
            [
                "통합제품ID",
                "이미지번호",
                "이미지파일",
                "다운로드상태",
                "OCR성공",
                "OCR전문",
                "오류",
            ]
        ].head(15)
    )


print()
print("=" * 70)
print("🎉 전체 작업 완료")
print("=" * 70)


# ============================================================
# 15. Excel 자동 다운로드
# ============================================================

print()
print("=" * 70)
print("⬇️ Excel 자동 다운로드 시작")
print("=" * 70)

try:

    from google.colab import files

    files.download(
        OUTPUT_FILE
    )

    print()
    print("✅ 다운로드 요청 완료")
    print("파일 :", OUTPUT_FILE)

except Exception as e:

    print()
    print("⚠️ 자동 다운로드 실패")
    print("오류 :", e)
    print()
    print("파일은 다음 경로에 저장되어 있습니다.")
    print(OUTPUT_FILE)

In [ ]:
# ============================================================
# CELL 7
# 🤖 Gemini OCR 의미 해석
#
# 기능
# 1. OCR전문 읽기
# 2. Gemini 3.5 Flash-Lite로 이미지별 OCR 의미 해석
# 3. Unicode / ASCII 인코딩 오류 방지
# 4. OCR 오류의 의미적 보정
# 5. 제조사 / 수입원 / 판매자 구분
# 6. 원재료 / 알레르기 / 마케팅 소구 / 용량 / 중량 / 수량 등 추출
# 7. 이미지별 체크포인트 저장
# 8. 중단 후 자동 이어하기
# 9. 상품 단위로 최종 통합
# 10. 최종 Excel 자동 다운로드
#
# 기존 Cell 3~6은 변경하지 않습니다.
# ============================================================


# ============================================================
# 0. 필요한 패키지만 설치
# ============================================================

print("=" * 70)
print("📦 Gemini SDK 준비")
print("=" * 70)

# pandas / google-auth를 강제로 업그레이드하지 않음
!pip -q install -U google-genai openpyxl


# ============================================================
# 1. 기본 라이브러리
# ============================================================

import os
import re
import json
import time
import math
import unicodedata
import traceback
import getpass

import pandas as pd
import numpy as np

from pathlib import Path


# ============================================================
# 2. Google Gemini
# ============================================================

from google import genai
from google.genai import types


# ============================================================
# 3. 경로 설정
# ============================================================

BASE_DIR = "/content/coupang_ocr"

RESULT_DIR = os.path.join(
    BASE_DIR,
    "results"
)

INPUT_XLSX = os.path.join(
    RESULT_DIR,
    "coupang_ocr_raw.xlsx"
)

OUTPUT_XLSX = os.path.join(
    RESULT_DIR,
    "coupang_ocr_gemini.xlsx"
)

CHECKPOINT_XLSX = os.path.join(
    RESULT_DIR,
    "coupang_ocr_gemini_checkpoint.xlsx"
)

os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================
# 4. Gemini 모델
# ============================================================

MODEL = "gemini-3.5-flash-lite"


print()
print("=" * 70)
print("🤖 Gemini OCR 의미 해석 시작")
print("=" * 70)

print()
print(f"📂 입력 파일 : {INPUT_XLSX}")
print(f"📂 출력 파일 : {OUTPUT_XLSX}")
print(f"💾 체크포인트 : {CHECKPOINT_XLSX}")
print(f"🤖 모델 : {MODEL}")


# ============================================================
# 5. API Key
# ============================================================

print()
print("=" * 70)
print("🔑 Gemini API Key")
print("=" * 70)

print()
print("Google AI Studio에서 발급한 Gemini API Key를 사용합니다.")
print()
print("가능하면 Colab Secrets에")
print("GEMINI_API_KEY")
print("이름으로 저장하세요.")
print()

API_KEY = None


# ------------------------------------------------------------
# 5-1. 환경변수
# ------------------------------------------------------------

if os.environ.get("GEMINI_API_KEY"):
    API_KEY = os.environ.get("GEMINI_API_KEY")
    print("✅ 환경변수 GEMINI_API_KEY 사용")


# ------------------------------------------------------------
# 5-2. Colab Secrets
# ------------------------------------------------------------

if not API_KEY:

    try:
        from google.colab import userdata

        try:
            API_KEY = userdata.get("GEMINI_API_KEY")
        except Exception:
            API_KEY = None

        if API_KEY:
            print("✅ Colab Secrets의 GEMINI_API_KEY 사용")

    except Exception:
        pass


# ------------------------------------------------------------
# 5-3. 직접 입력
# ------------------------------------------------------------

if not API_KEY:

    API_KEY = getpass.getpass(
        "Gemini API Key를 입력하세요: "
    )

    if API_KEY:
        print("✅ Gemini API Key 입력 완료")
    else:
        raise ValueError(
            "Gemini API Key가 입력되지 않았습니다."
        )


# ============================================================
# 6. Gemini Client
# ============================================================

client = genai.Client(
    api_key=API_KEY
)

print()
print(f"🤖 모델 : {MODEL}")


# ============================================================
# 7. Excel 읽기
# ============================================================

print()
print("=" * 70)
print("📖 OCR Excel 읽는 중")
print("=" * 70)

if not os.path.exists(INPUT_XLSX):
    raise FileNotFoundError(
        f"입력 파일을 찾을 수 없습니다:\n{INPUT_XLSX}"
    )

xls = pd.ExcelFile(INPUT_XLSX)

print("시트 :", xls.sheet_names)


ocr_detail = pd.read_excel(
    INPUT_XLSX,
    sheet_name="OCR전문"
)

ocr_result = pd.read_excel(
    INPUT_XLSX,
    sheet_name="OCR결과"
)

product_master = pd.read_excel(
    INPUT_XLSX,
    sheet_name="ProductMaster"
)


print()
print(f"OCR전문 행 수 : {len(ocr_detail):,}")
print(f"OCR결과 행 수 : {len(ocr_result):,}")
print(f"ProductMaster 행 수 : {len(product_master):,}")


# ============================================================
# 8. 컬럼 확인
# ============================================================

print()
print("=" * 70)
print("🔎 OCR 컬럼 확인")
print("=" * 70)

print()
print("OCR전문 컬럼:")
print(list(ocr_detail.columns))

print()
print("ProductMaster 컬럼:")
print(list(product_master.columns))


# ------------------------------------------------------------
# 컬럼 자동 탐색
# ------------------------------------------------------------

def find_column(df, candidates):

    for col in candidates:
        if col in df.columns:
            return col

    return None


OCR_TEXT_COL = find_column(
    ocr_detail,
    [
        "OCR전문",
        "OCR텍스트",
        "OCR",
        "텍스트"
    ]
)

IMAGE_NO_COL = find_column(
    ocr_detail,
    [
        "이미지번호",
        "이미지 번호",
        "image_no"
    ]
)

IMAGE_FILE_COL = find_column(
    ocr_detail,
    [
        "이미지파일",
        "이미지 파일",
        "image_file"
    ]
)

PRODUCT_ID_COL = find_column(
    ocr_detail,
    [
        "통합제품ID",
        "제품ID",
        "상품ID"
    ]
)

PRODUCT_NAME_COL = find_column(
    ocr_detail,
    [
        "상품명",
        "제품명"
    ]
)


if OCR_TEXT_COL is None:
    raise ValueError(
        "OCR 텍스트 컬럼을 찾을 수 없습니다."
    )

if PRODUCT_ID_COL is None:
    raise ValueError(
        "통합제품ID 컬럼을 찾을 수 없습니다."
    )

if IMAGE_NO_COL is None:
    raise ValueError(
        "이미지번호 컬럼을 찾을 수 없습니다."
    )


print()
print("OCR 텍스트 컬럼 :", OCR_TEXT_COL)
print("이미지 번호 컬럼 :", IMAGE_NO_COL)
print("이미지 파일 컬럼 :", IMAGE_FILE_COL)
print("제품 ID 컬럼 :", PRODUCT_ID_COL)


# ============================================================
# 9. Product Master dictionary
# ============================================================

master_lookup = {}

for _, row in product_master.iterrows():

    pid = str(
        row.get("통합제품ID", "")
    ).strip()

    if not pid:
        continue

    master_lookup[pid] = {
        "상품URL": row.get("상품URL", ""),
        "브랜드": row.get("브랜드", ""),
        "상품명": row.get("상품명", ""),
        "itemId": row.get("itemId", ""),
        "vendorItemId": row.get("vendorItemId", "")
    }


print()
print(f"📦 Product Master 등록 : {len(master_lookup):,}개")


# ============================================================
# 10. Unicode 안전 처리
# ============================================================

def safe_text(value):
    """
    Gemini API로 보내기 전 문자열을 안전하게 정제.
    OCR 특유의 깨진 Unicode / surrogate / 제어문자 제거.
    """

    if value is None:
        return ""

    try:
        text = str(value)
    except Exception:
        return ""

    # --------------------------------------------------------
    # Unicode 정규화
    # --------------------------------------------------------

    try:
        text = unicodedata.normalize(
            "NFKC",
            text
        )
    except Exception:
        pass

    # --------------------------------------------------------
    # UTF-16 surrogate 제거
    # --------------------------------------------------------

    try:
        text = re.sub(
            r"[\ud800-\udfff]",
            "",
            text
        )
    except Exception:
        pass

    # --------------------------------------------------------
    # 제어문자 제거
    # \n \r \t 제외
    # --------------------------------------------------------

    cleaned_chars = []

    for ch in text:

        try:
            category = unicodedata.category(ch)

            if (
                ch in "\n\r\t"
                or not category.startswith("C")
            ):
                cleaned_chars.append(ch)

        except Exception:
            continue

    text = "".join(cleaned_chars)

    # --------------------------------------------------------
    # UTF-8 round trip
    # --------------------------------------------------------

    try:

        text = (
            text
            .encode(
                "utf-8",
                errors="ignore"
            )
            .decode(
                "utf-8",
                errors="ignore"
            )
        )

    except Exception:
        pass

    # --------------------------------------------------------
    # 공백 정리
    # --------------------------------------------------------

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# ============================================================
# 11. OCR DataFrame 전처리
# ============================================================

ocr_detail = ocr_detail.copy()

ocr_detail["_OCR_SAFE"] = (
    ocr_detail[OCR_TEXT_COL]
    .fillna("")
    .map(safe_text)
)

ocr_detail["_PRODUCT_ID_SAFE"] = (
    ocr_detail[PRODUCT_ID_COL]
    .fillna("")
    .map(safe_text)
)


# ============================================================
# 12. Gemini Image Schema
# ============================================================

IMAGE_SCHEMA = {
    "type": "object",
    "properties": {

        "product_name": {
            "type": "string",
            "description": "이미지에서 확인되는 정확한 제품명. 없으면 미확인."
        },

        "manufacturer": {
            "type": "string",
            "description": "제조사. 제조업체가 명시되지 않으면 미확인."
        },

        "manufacturer_address": {
            "type": "string",
            "description": "제조사 주소. 없으면 미확인."
        },

        "importer": {
            "type": "string",
            "description": "수입원 또는 수입판매원. 없으면 미확인."
        },

        "seller": {
            "type": "string",
            "description": "판매자. 제조사와 구분. 없으면 미확인."
        },

        "ingredients": {
            "type": "string",
            "description": "원재료명 및 성분. 없으면 미확인."
        },

        "allergens": {
            "type": "string",
            "description": "명시된 알레르기 유발물질. 없으면 미확인."
        },

        "marketing_appeal": {
            "type": "string",
            "description": "이미지에 실제로 명시된 마케팅 소구 포인트. 여러 개면 / 로 구분."
        },

        "size": {
            "type": "string",
            "description": "제품의 사이즈. 없으면 미확인."
        },

        "capacity": {
            "type": "string",
            "description": "용량. 예: 195ml, 340ml. 없으면 미확인."
        },

        "weight": {
            "type": "string",
            "description": "개당 중량. 예: 27g/개. 없으면 미확인."
        },

        "total_weight": {
            "type": "string",
            "description": "전체 패키지 총중량. 없으면 미확인."
        },

        "quantity": {
            "type": "string",
            "description": "전체 수량 또는 낱개 수. 명시되지 않으면 미확인."
        },

        "pack_count": {
            "type": "string",
            "description": "한 포장에 들어 있는 매수/개수. 명시되지 않으면 미확인."
        },

        "bundle_count": {
            "type": "string",
            "description": "묶음 수. 명시되지 않으면 미확인."
        },

        "country_of_origin": {
            "type": "string",
            "description": "원산지. 없으면 미확인."
        },

        "packaging_material": {
            "type": "string",
            "description": "포장재질. 없으면 미확인."
        },

        "storage": {
            "type": "string",
            "description": "보관방법. 없으면 미확인."
        },

        "expiration": {
            "type": "string",
            "description": "소비기한/유통기한 관련 정보. 없으면 미확인."
        },

        "usage": {
            "type": "string",
            "description": "사용방법 또는 섭취방법. 없으면 미확인."
        },

        "components": {
            "type": "string",
            "description": "구성품. 없으면 미확인."
        },

        "other_information": {
            "type": "string",
            "description": "위 항목에 포함되지 않는 중요한 제품 정보."
        },

        "evidence": {
            "type": "string",
            "description": "판단 근거가 된 OCR 문구. 중요한 경우 원문 일부를 유지."
        },

        "confidence": {
            "type": "string",
            "description": "높음 / 중간 / 낮음"
        }
    },

    "required": [
        "product_name",
        "manufacturer",
        "manufacturer_address",
        "importer",
        "seller",
        "ingredients",
        "allergens",
        "marketing_appeal",
        "size",
        "capacity",
        "weight",
        "total_weight",
        "quantity",
        "pack_count",
        "bundle_count",
        "country_of_origin",
        "packaging_material",
        "storage",
        "expiration",
        "usage",
        "components",
        "other_information",
        "evidence",
        "confidence"
    ]
}


# ============================================================
# 13. Gemini 이미지 해석 함수
# ============================================================

def gemini_extract_image(
    client,
    model,
    product_id,
    image_no,
    master_product_name,
    ocr_text,
    schema,
    max_retries=4
):

    # --------------------------------------------------------
    # 안전 문자열
    # --------------------------------------------------------

    product_id = safe_text(product_id)
    image_no = safe_text(image_no)
    master_product_name = safe_text(
        master_product_name
    )

    ocr_text = safe_text(
        ocr_text
    )

    # --------------------------------------------------------
    # OCR 없음
    # --------------------------------------------------------

    if not ocr_text:

        return {
            "status": "EMPTY",
            "data": {},
            "raw": "",
            "error": "OCR 텍스트 없음"
        }

    # --------------------------------------------------------
    # 너무 긴 OCR은 안전하게 제한
    # --------------------------------------------------------

    # 제품 상세 이미지 OCR은 지나치게 긴 경우
    # API 요청 크기를 줄이기 위해 최대 30,000자 사용
    if len(ocr_text) > 30000:

        ocr_text = ocr_text[:30000]

        ocr_text += (
            "\n\n[OCR 텍스트가 30,000자 이후는 생략됨]"
        )

    # --------------------------------------------------------
    # Prompt
    # --------------------------------------------------------

    prompt = f"""
당신은 한국 온라인 쇼핑몰 상품 상세페이지의 OCR 데이터를
정확한 상품정보로 정리하는 데이터 추출 전문가입니다.

아래 OCR은 쿠팡 상품 상세 이미지에서 추출된 텍스트입니다.

OCR에는 다음과 같은 문제가 있을 수 있습니다.

- 한글 OCR 오류
- 영문 OCR 오류
- 숫자 OCR 오류
- 띄어쓰기 오류
- 제품명 왜곡
- 중복 문장
- 특수문자 오류
- 제조사명 왜곡
- 브랜드명 왜곡

OCR을 단순 복사하지 말고 문맥을 이용하여 의미를 해석하십시오.

그러나 반드시 다음 원칙을 지키십시오.

==================================================
[핵심 원칙]
==================================================

1. OCR에 실제로 존재하는 정보만 사용합니다.

2. 정보가 없거나 확실하게 확인할 수 없으면
   반드시 "미확인"이라고 작성합니다.

3. 절대로 숫자나 수량을 추정하지 않습니다.

4. "180g/개"처럼 표시된 경우
   이것은 개당 중량이며 수량이 아닙니다.

5. "총 60개입"처럼 명확하게 표시된 경우에만
   수량을 60개로 기록합니다.

6. "대용량", "묶음", "세트" 등의 표현만으로
   실제 개수를 추정하지 않습니다.

7. 제조사 / 수입원 / 판매자를 반드시 구분합니다.

8. 한국마즈처럼 수입원으로 명시된 업체는
   제조사로 기록하지 않습니다.

9. 판매자 또는 고객센터 정보는 제조사로 기록하지 않습니다.

10. 마케팅 소구 포인트는 이미지에 실제로 표현된
    문구 또는 명확한 제품 특성만 기록합니다.

11. OCR 오류가 명백하고 문맥상 수정이 확실한 경우에만
    제품명을 보정합니다.

12. 보정이 확실하지 않으면 원문을 유지하거나 "미확인"으로 합니다.

13. 여러 제품이 하나의 이미지에 등장하는 경우
    해당 이미지에서 확인되는 정보를 종합하여 기록합니다.

14. 원재료와 알레르기 정보는 가능한 한 원문에 충실하게 기록합니다.

15. 제조사 주소는 제조사 주소가 명시된 경우에만 기록합니다.

16. 원산지와 제조국은 명시된 경우에만 기록합니다.

17. 소비기한/유통기한은 실제 표시가 있을 때만 기록합니다.

==================================================
[제품 정보]
==================================================

통합제품ID:
{product_id}

Product Master 상품명:
{master_product_name}

이미지 번호:
{image_no}

==================================================
[OCR TEXT]
==================================================

{ocr_text}

==================================================
[출력]
==================================================

반드시 지정된 JSON 형식으로만 반환하십시오.
"""

    prompt = safe_text(prompt)

    last_error = ""

    # --------------------------------------------------------
    # API Retry
    # --------------------------------------------------------

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0
                )
            )

            # ------------------------------------------------
            # 응답 추출
            # ------------------------------------------------

            response_text = getattr(
                response,
                "text",
                None
            )

            if response_text is None:
                response_text = ""

            response_text = safe_text(
                response_text
            )

            if not response_text:
                raise ValueError(
                    "Gemini 응답이 비어 있습니다."
                )

            # ------------------------------------------------
            # JSON 파싱
            # ------------------------------------------------

            try:

                data = json.loads(
                    response_text
                )

            except Exception:

                # 혹시 Markdown JSON이면 제거
                cleaned = response_text

                cleaned = re.sub(
                    r"^```json\s*",
                    "",
                    cleaned,
                    flags=re.IGNORECASE
                )

                cleaned = re.sub(
                    r"\s*```$",
                    "",
                    cleaned
                )

                data = json.loads(
                    cleaned
                )

            return {
                "status": "SUCCESS",
                "data": data,
                "raw": response_text,
                "error": ""
            }

        except Exception as e:

            last_error = str(e)

            print(
                f"      ⚠️ Gemini 오류 "
                f"(시도 {attempt}/{max_retries}): "
                f"{last_error}"
            )

            error_lower = last_error.lower()

            # ------------------------------------------------
            # Unicode / ASCII 오류
            # ------------------------------------------------

            encoding_error = (
                "ascii" in error_lower
                or "unicode" in error_lower
                or "codec can't encode" in error_lower
                or "codec can't decode" in error_lower
                or "surrogate" in error_lower
            )

            if encoding_error:

                print(
                    "      🔧 Unicode/인코딩 오류 감지"
                )

                print(
                    "      → OCR 텍스트 재정제 후 재시도"
                )

                ocr_text = safe_text(
                    ocr_text
                )

                prompt = safe_text(
                    prompt
                )

                if attempt < max_retries:

                    time.sleep(2)

                    continue

            # ------------------------------------------------
            # 일반 오류
            # ------------------------------------------------

            if attempt < max_retries:

                wait_sec = min(
                    5 * (2 ** (attempt - 1)),
                    60
                )

                print(
                    f"      ⏳ "
                    f"{wait_sec}초 후 재시도..."
                )

                time.sleep(
                    wait_sec
                )

    return {
        "status": "ERROR",
        "data": {},
        "raw": "",
        "error": last_error
    }


# ============================================================
# 14. 기존 체크포인트 불러오기
# ============================================================

processed_results = []


if os.path.exists(
    CHECKPOINT_XLSX
):

    print()
    print("=" * 70)
    print("💾 기존 체크포인트 발견")
    print("=" * 70)

    try:

        checkpoint_df = pd.read_excel(
            CHECKPOINT_XLSX,
            sheet_name="AI_이미지해석"
        )

        processed_results = (
            checkpoint_df
            .to_dict("records")
        )

        print(
            f"기존 체크포인트 행 수 : "
            f"{len(processed_results):,}"
        )

    except Exception as e:

        print(
            f"⚠️ 체크포인트 읽기 실패: {e}"
        )

        processed_results = []

else:

    print()
    print("ℹ️ 기존 체크포인트 없음")
    print("   처음부터 처리합니다.")


# ============================================================
# 15. 이미 처리된 이미지 Key
# ============================================================

processed_keys = set()

for row in processed_results:

    key = (
        safe_text(
            row.get("통합제품ID", "")
        ),
        safe_text(
            row.get("이미지번호", "")
        )
    )

    processed_keys.add(key)


print(
    f"이미 처리된 이미지 : "
    f"{len(processed_keys):,}"
)


# ============================================================
# 16. 처리 대상 확인
# ============================================================

total_images = len(
    ocr_detail
)

ocr_exists_count = int(
    (
        ocr_detail["_OCR_SAFE"]
        .str.strip()
        != ""
    ).sum()
)

ocr_empty_count = (
    total_images
    - ocr_exists_count
)


print()
print("=" * 70)
print("📊 처리 대상")
print("=" * 70)

print(
    f"전체 OCR 이미지 : "
    f"{total_images:,}"
)

print(
    f"OCR 텍스트 있음 : "
    f"{ocr_exists_count:,}"
)

print(
    f"OCR 텍스트 없음 : "
    f"{ocr_empty_count:,}"
)


# ============================================================
# 17. STEP 1
# 이미지별 Gemini 해석
# ============================================================

print()
print("=" * 70)
print("🔎 STEP 1. 이미지별 Gemini OCR 해석")
print("=" * 70)


new_results = []

processed_this_run = 0


for idx, row in ocr_detail.iterrows():

    product_id = safe_text(
        row.get(
            PRODUCT_ID_COL,
            ""
        )
    )

    image_no = safe_text(
        row.get(
            IMAGE_NO_COL,
            ""
        )
    )

    image_file = ""

    if IMAGE_FILE_COL:

        image_file = safe_text(
            row.get(
                IMAGE_FILE_COL,
                ""
            )
        )

    ocr_text = safe_text(
        row.get(
            "_OCR_SAFE",
            ""
        )
    )

    # --------------------------------------------------------
    # Product Master
    # --------------------------------------------------------

    master_info = master_lookup.get(
        product_id,
        {}
    )

    master_product_name = safe_text(
        master_info.get(
            "상품명",
            row.get("상품명", "")
        )
    )

    master_brand = safe_text(
        master_info.get(
            "브랜드",
            row.get("브랜드", "")
        )
    )

    product_url = safe_text(
        master_info.get(
            "상품URL",
            row.get("상품URL", "")
        )
    )

    vendor_item_id = safe_text(
        row.get(
            "vendorItemId",
            master_info.get(
                "vendorItemId",
                ""
            )
        )
    )

    item_id = safe_text(
        row.get(
            "itemId",
            master_info.get(
                "itemId",
                ""
            )
        )
    )

    html_file = safe_text(
        row.get(
            "HTML파일",
            ""
        )
    )

    image_url = safe_text(
        row.get(
            "이미지URL",
            ""
        )
    )

    download_status = safe_text(
        row.get(
            "다운로드상태",
            ""
        )
    )

    # --------------------------------------------------------
    # 처리 Key
    # --------------------------------------------------------

    key = (
        product_id,
        image_no
    )

    # --------------------------------------------------------
    # 이미 처리된 이미지
    # --------------------------------------------------------

    if key in processed_keys:

        continue

    # --------------------------------------------------------
    # OCR 없음
    # --------------------------------------------------------

    if not ocr_text:

        result_row = {
            "통합제품ID": product_id,
            "상품URL": product_url,
            "브랜드": master_brand,
            "상품명": master_product_name,
            "vendorItemId": vendor_item_id,
            "itemId": item_id,
            "HTML파일": html_file,
            "이미지번호": image_no,
            "이미지URL": image_url,
            "이미지파일": image_file,
            "다운로드상태": download_status,
            "AI상태": "EMPTY",
            "AI상품명": "미확인",
            "제조사": "미확인",
            "제조사주소": "미확인",
            "수입원": "미확인",
            "판매자": "미확인",
            "원재료": "미확인",
            "알레르기": "미확인",
            "마케팅소구포인트": "미확인",
            "사이즈": "미확인",
            "용량": "미확인",
            "개당중량": "미확인",
            "총중량": "미확인",
            "수량": "미확인",
            "포장당매수": "미확인",
            "묶음수": "미확인",
            "원산지": "미확인",
            "포장재질": "미확인",
            "보관방법": "미확인",
            "소비기한": "미확인",
            "사용방법": "미확인",
            "구성품": "미확인",
            "기타정보": "미확인",
            "근거": "OCR 텍스트 없음",
            "신뢰도": "낮음",
            "OCR전문": "",
            "AI_RAW_JSON": "",
            "오류": "OCR 텍스트 없음"
        }

        new_results.append(
            result_row
        )

        processed_keys.add(key)

        continue

    # --------------------------------------------------------
    # 진행 표시
    # --------------------------------------------------------

    print(
        f"📦 {product_id} | "
        f"이미지 {image_no}: 🤖 해석 중..."
    )

    # --------------------------------------------------------
    # Gemini
    # --------------------------------------------------------

    result = gemini_extract_image(
        client=client,
        model=MODEL,
        product_id=product_id,
        image_no=image_no,
        master_product_name=master_product_name,
        ocr_text=ocr_text,
        schema=IMAGE_SCHEMA
    )

    data = result.get(
        "data",
        {}
    )

    # --------------------------------------------------------
    # 결과 추출
    # --------------------------------------------------------

    result_row = {
        "통합제품ID": product_id,
        "상품URL": product_url,
        "브랜드": master_brand,
        "상품명": master_product_name,
        "vendorItemId": vendor_item_id,
        "itemId": item_id,
        "HTML파일": html_file,
        "이미지번호": image_no,
        "이미지URL": image_url,
        "이미지파일": image_file,
        "다운로드상태": download_status,

        "AI상태": result.get(
            "status",
            "ERROR"
        ),

        "AI상품명": data.get(
            "product_name",
            "미확인"
        ),

        "제조사": data.get(
            "manufacturer",
            "미확인"
        ),

        "제조사주소": data.get(
            "manufacturer_address",
            "미확인"
        ),

        "수입원": data.get(
            "importer",
            "미확인"
        ),

        "판매자": data.get(
            "seller",
            "미확인"
        ),

        "원재료": data.get(
            "ingredients",
            "미확인"
        ),

        "알레르기": data.get(
            "allergens",
            "미확인"
        ),

        "마케팅소구포인트": data.get(
            "marketing_appeal",
            "미확인"
        ),

        "사이즈": data.get(
            "size",
            "미확인"
        ),

        "용량": data.get(
            "capacity",
            "미확인"
        ),

        "개당중량": data.get(
            "weight",
            "미확인"
        ),

        "총중량": data.get(
            "total_weight",
            "미확인"
        ),

        "수량": data.get(
            "quantity",
            "미확인"
        ),

        "포장당매수": data.get(
            "pack_count",
            "미확인"
        ),

        "묶음수": data.get(
            "bundle_count",
            "미확인"
        ),

        "원산지": data.get(
            "country_of_origin",
            "미확인"
        ),

        "포장재질": data.get(
            "packaging_material",
            "미확인"
        ),

        "보관방법": data.get(
            "storage",
            "미확인"
        ),

        "소비기한": data.get(
            "expiration",
            "미확인"
        ),

        "사용방법": data.get(
            "usage",
            "미확인"
        ),

        "구성품": data.get(
            "components",
            "미확인"
        ),

        "기타정보": data.get(
            "other_information",
            "미확인"
        ),

        "근거": data.get(
            "evidence",
            "미확인"
        ),

        "신뢰도": data.get(
            "confidence",
            "낮음"
        ),

        "OCR전문": ocr_text,

        "AI_RAW_JSON": json.dumps(
            data,
            ensure_ascii=False
        ),

        "오류": result.get(
            "error",
            ""
        )
    }

    new_results.append(
        result_row
    )

    processed_keys.add(key)

    processed_this_run += 1

    # --------------------------------------------------------
    # 이미지 사이 API 요청 간격
    # --------------------------------------------------------

    time.sleep(2)

    # --------------------------------------------------------
    # 5개마다 체크포인트
    # --------------------------------------------------------

    if (
        processed_this_run % 5 == 0
    ):

        print()
        print(
            f"💾 체크포인트 저장 "
            f"({processed_this_run}개 추가)"
        )

        checkpoint_all = (
            processed_results
            + new_results
        )

        checkpoint_df = pd.DataFrame(
            checkpoint_all
        )

        with pd.ExcelWriter(
            CHECKPOINT_XLSX,
            engine="openpyxl"
        ) as writer:

            checkpoint_df.to_excel(
                writer,
                sheet_name="AI_이미지해석",
                index=False
            )

        print(
            f"   → {CHECKPOINT_XLSX}"
        )

        print()


# ============================================================
# 18. 이미지별 결과 합치기
# ============================================================

all_image_results = (
    processed_results
    + new_results
)


image_ai_df = pd.DataFrame(
    all_image_results
)


# ------------------------------------------------------------
# 중복 방지
# ------------------------------------------------------------

if len(image_ai_df) > 0:

    image_ai_df = (
        image_ai_df
        .drop_duplicates(
            subset=[
                "통합제품ID",
                "이미지번호"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )


print()
print("=" * 70)
print("✅ STEP 1 완료")
print("=" * 70)

print(
    f"이미지별 AI 해석 결과 : "
    f"{len(image_ai_df):,}개"
)

if "AI상태" in image_ai_df.columns:

    print(
        image_ai_df["AI상태"]
        .value_counts(dropna=False)
    )


# ============================================================
# 19. 상품 통합 Schema
# ============================================================

PRODUCT_SCHEMA = {
    "type": "object",

    "properties": {

        "product_name": {
            "type": "string"
        },

        "components": {
            "type": "string"
        },

        "manufacturer": {
            "type": "string"
        },

        "importer": {
            "type": "string"
        },

        "ingredients": {
            "type": "string"
        },

        "allergens": {
            "type": "string"
        },

        "marketing_appeal": {
            "type": "string"
        },

        "size": {
            "type": "string"
        },

        "capacity": {
            "type": "string"
        },

        "weight": {
            "type": "string"
        },

        "total_weight": {
            "type": "string"
        },

        "quantity": {
            "type": "string"
        },

        "pack_count": {
            "type": "string"
        },

        "bundle_count": {
            "type": "string"
        },

        "country_of_origin": {
            "type": "string"
        },

        "packaging_material": {
            "type": "string"
        },

        "storage": {
            "type": "string"
        },

        "expiration": {
            "type": "string"
        },

        "usage": {
            "type": "string"
        },

        "other_information": {
            "type": "string"
        },

        "evidence_images": {
            "type": "string"
        },

        "needs_review": {
            "type": "string"
        },

        "confidence": {
            "type": "string"
        }
    },

    "required": [
        "product_name",
        "components",
        "manufacturer",
        "importer",
        "ingredients",
        "allergens",
        "marketing_appeal",
        "size",
        "capacity",
        "weight",
        "total_weight",
        "quantity",
        "pack_count",
        "bundle_count",
        "country_of_origin",
        "packaging_material",
        "storage",
        "expiration",
        "usage",
        "other_information",
        "evidence_images",
        "needs_review",
        "confidence"
    ]
}


# ============================================================
# 20. 상품 단위 Gemini 통합 함수
# ============================================================

def gemini_integrate_product(
    client,
    model,
    product_id,
    brand,
    master_name,
    image_rows,
    schema,
    max_retries=4
):

    product_id = safe_text(
        product_id
    )

    brand = safe_text(
        brand
    )

    master_name = safe_text(
        master_name
    )

    # --------------------------------------------------------
    # 이미지별 AI 결과를 텍스트로 구성
    # --------------------------------------------------------

    image_blocks = []

    for _, r in image_rows.iterrows():

        image_no = safe_text(
            r.get(
                "이미지번호",
                ""
            )
        )

        ai_raw = safe_text(
            r.get(
                "AI_RAW_JSON",
                ""
            )
        )

        ocr_text = safe_text(
            r.get(
                "OCR전문",
                ""
            )
        )

        if not ai_raw:
            ai_raw = "{}"

        block = f"""
[이미지 {image_no}]

AI 이미지 해석:
{ai_raw}

원본 OCR:
{ocr_text[:12000]}
"""

        image_blocks.append(
            block
        )

    combined = "\n\n".join(
        image_blocks
    )

    # --------------------------------------------------------
    # 너무 길 경우 제한
    # --------------------------------------------------------

    if len(combined) > 50000:

        combined = combined[:50000]

        combined += (
            "\n\n[추가 OCR 데이터는 생략됨]"
        )

    # --------------------------------------------------------
    # Prompt
    # --------------------------------------------------------

    prompt = f"""
당신은 온라인 쇼핑몰 상품 데이터 정리 전문가입니다.

하나의 상품에 대한 여러 상세 이미지의 OCR 및 AI 해석 결과를
통합하여 하나의 정확한 상품 정보를 만들어야 합니다.

==================================================
상품 기본정보
==================================================

통합제품ID:
{product_id}

브랜드:
{brand}

Product Master 상품명:
{master_name}

==================================================
통합 규칙
==================================================

1. 여러 이미지에서 동일하게 확인되는 정보는 하나로 통합합니다.

2. 한 이미지에만 존재하는 정보라도 명확하면 포함합니다.

3. 서로 충돌하는 정보가 있으면 더 명확한 원문을 우선합니다.

4. 제조사 / 수입원 / 판매자를 구분합니다.

5. 제조사와 수입원을 절대로 합치지 않습니다.

6. 수량은 실제로 명시된 경우에만 기록합니다.

7. "180g/개"는 개당중량입니다.
   수량으로 해석하지 않습니다.

8. "총 60개입"처럼 명시된 경우에만
   수량을 60개로 기록합니다.

9. 수량이 명확하지 않으면 "미확인"입니다.

10. 포장당 매수와 묶음수 역시 실제 명시된 경우에만 기록합니다.

11. 마케팅 소구 포인트는 실제 상품 이미지에 표현된
    내용을 기반으로 작성합니다.

12. OCR에 없는 내용을 추정하지 않습니다.

13. 서로 다른 제품명이 보이는 경우
    Product Master 상품명과 이미지 정보를 함께 비교합니다.

14. 제품명은 명확하게 확인되는 경우만 보정합니다.

15. 확인이 불가능하면 "미확인"입니다.

==================================================
이미지별 데이터
==================================================

{combined}

==================================================
출력
==================================================

반드시 지정된 JSON 형식으로만 반환하십시오.
"""

    prompt = safe_text(
        prompt
    )

    last_error = ""

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0
                )
            )

            response_text = safe_text(
                getattr(
                    response,
                    "text",
                    ""
                )
            )

            if not response_text:
                raise ValueError(
                    "Gemini 상품 통합 응답이 비어 있습니다."
                )

            try:

                data = json.loads(
                    response_text
                )

            except Exception:

                cleaned = re.sub(
                    r"^```json\s*",
                    "",
                    response_text,
                    flags=re.IGNORECASE
                )

                cleaned = re.sub(
                    r"\s*```$",
                    "",
                    cleaned
                )

                data = json.loads(
                    cleaned
                )

            return {
                "status": "SUCCESS",
                "data": data,
                "raw": response_text,
                "error": ""
            }

        except Exception as e:

            last_error = str(e)

            print(
                f"      ⚠️ 상품 통합 오류 "
                f"(시도 {attempt}/{max_retries}): "
                f"{last_error}"
            )

            if attempt < max_retries:

                wait_sec = min(
                    5 * (2 ** (attempt - 1)),
                    60
                )

                print(
                    f"      ⏳ "
                    f"{wait_sec}초 후 재시도..."
                )

                time.sleep(
                    wait_sec
                )

    return {
        "status": "ERROR",
        "data": {},
        "raw": "",
        "error": last_error
    }


# ============================================================
# 21. STEP 2
# 상품 단위 통합
# ============================================================

print()
print("=" * 70)
print("🔎 STEP 2. 상품 단위 Gemini 정보 통합")
print("=" * 70)


product_results = []


if len(image_ai_df) > 0:

    product_groups = (
        image_ai_df
        .groupby(
            "통합제품ID",
            sort=False
        )
    )

    total_products = (
        image_ai_df[
            "통합제품ID"
        ]
        .nunique()
    )

    print(
        f"통합 대상 상품 수 : "
        f"{total_products:,}"
    )

    for product_idx, (
        product_id,
        group
    ) in enumerate(
        product_groups,
        start=1
    ):

        product_id = safe_text(
            product_id
        )

        master_info = master_lookup.get(
            product_id,
            {}
        )

        brand = safe_text(
            master_info.get(
                "브랜드",
                group.iloc[0].get(
                    "브랜드",
                    ""
                )
            )
        )

        master_name = safe_text(
            master_info.get(
                "상품명",
                group.iloc[0].get(
                    "상품명",
                    ""
                )
            )
        )

        print()
        print(
            f"📦 [{product_idx}/{total_products}] "
            f"{product_id}"
        )

        print(
            f"   상품명 : {master_name}"
        )

        print(
            f"   이미지 수 : {len(group):,}"
        )

        result = gemini_integrate_product(
            client=client,
            model=MODEL,
            product_id=product_id,
            brand=brand,
            master_name=master_name,
            image_rows=group,
            schema=PRODUCT_SCHEMA
        )

        data = result.get(
            "data",
            {}
        )

        product_row = {

            "통합제품ID": product_id,

            "브랜드": brand,

            "상품명_Master": master_name,

            "AI상품명": data.get(
                "product_name",
                "미확인"
            ),

            "구성품": data.get(
                "components",
                "미확인"
            ),

            "제조사": data.get(
                "manufacturer",
                "미확인"
            ),

            "수입원": data.get(
                "importer",
                "미확인"
            ),

            "원재료": data.get(
                "ingredients",
                "미확인"
            ),

            "알레르기": data.get(
                "allergens",
                "미확인"
            ),

            "마케팅소구포인트": data.get(
                "marketing_appeal",
                "미확인"
            ),

            "사이즈": data.get(
                "size",
                "미확인"
            ),

            "용량": data.get(
                "capacity",
                "미확인"
            ),

            "개당중량": data.get(
                "weight",
                "미확인"
            ),

            "총중량": data.get(
                "total_weight",
                "미확인"
            ),

            "수량": data.get(
                "quantity",
                "미확인"
            ),

            "포장당매수": data.get(
                "pack_count",
                "미확인"
            ),

            "묶음수": data.get(
                "bundle_count",
                "미확인"
            ),

            "원산지": data.get(
                "country_of_origin",
                "미확인"
            ),

            "포장재질": data.get(
                "packaging_material",
                "미확인"
            ),

            "보관방법": data.get(
                "storage",
                "미확인"
            ),

            "소비기한": data.get(
                "expiration",
                "미확인"
            ),

            "사용방법": data.get(
                "usage",
                "미확인"
            ),

            "기타정보": data.get(
                "other_information",
                "미확인"
            ),

            "근거이미지": data.get(
                "evidence_images",
                "미확인"
            ),

            "확인필요": data.get(
                "needs_review",
                "미확인"
            ),

            "신뢰도": data.get(
                "confidence",
                "낮음"
            ),

            "이미지수": len(group),

            "AI_RAW_JSON": json.dumps(
                data,
                ensure_ascii=False
            ),

            "처리상태": result.get(
                "status",
                "ERROR"
            ),

            "오류": result.get(
                "error",
                ""
            )
        }

        product_results.append(
            product_row
        )

        time.sleep(2)


product_ai_df = pd.DataFrame(
    product_results
)


# ============================================================
# 22. 오류 데이터
# ============================================================

if len(image_ai_df) > 0:

    error_df = image_ai_df[
        image_ai_df["AI상태"]
        .isin(
            [
                "ERROR",
                "EMPTY"
            ]
        )
    ].copy()

else:

    error_df = pd.DataFrame()


# ============================================================
# 23. 기존 원본 시트 정리
# ============================================================

# 원본 Excel의 모든 시트를 그대로 유지
original_sheets = {}

for sheet in xls.sheet_names:

    try:

        original_sheets[sheet] = pd.read_excel(
            INPUT_XLSX,
            sheet_name=sheet
        )

    except Exception:

        original_sheets[sheet] = pd.DataFrame()


# ============================================================
# 24. 최종 Excel 저장
# ============================================================

print()
print("=" * 70)
print("💾 최종 Excel 저장")
print("=" * 70)


with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # AI 이미지 해석
    # --------------------------------------------------------

    image_ai_df.to_excel(
        writer,
        sheet_name="AI_이미지해석",
        index=False
    )

    # --------------------------------------------------------
    # AI 상품 통합
    # --------------------------------------------------------

    product_ai_df.to_excel(
        writer,
        sheet_name="AI_상품통합",
        index=False
    )

    # --------------------------------------------------------
    # AI 오류
    # --------------------------------------------------------

    error_df.to_excel(
        writer,
        sheet_name="AI_오류",
        index=False
    )

    # --------------------------------------------------------
    # 기존 OCR결과
    # --------------------------------------------------------

    original_sheets[
        "OCR결과"
    ].to_excel(
        writer,
        sheet_name="OCR결과",
        index=False
    )

    # --------------------------------------------------------
    # 기존 OCR전문
    # --------------------------------------------------------

    original_sheets[
        "OCR전문"
    ].to_excel(
        writer,
        sheet_name="OCR전문",
        index=False
    )

    # --------------------------------------------------------
    # ProductMaster
    # --------------------------------------------------------

    original_sheets[
        "ProductMaster"
    ].to_excel(
        writer,
        sheet_name="ProductMaster",
        index=False
    )

    # --------------------------------------------------------
    # HTML분석
    # --------------------------------------------------------

    if "HTML분석" in original_sheets:

        original_sheets[
            "HTML분석"
        ].to_excel(
            writer,
            sheet_name="HTML분석",
            index=False
        )

    # --------------------------------------------------------
    # 이미지다운로드
    # --------------------------------------------------------

    if "이미지다운로드" in original_sheets:

        original_sheets[
            "이미지다운로드"
        ].to_excel(
            writer,
            sheet_name="이미지다운로드",
            index=False
        )

    # --------------------------------------------------------
    # HTML매칭실패
    # --------------------------------------------------------

    if "HTML매칭실패" in original_sheets:

        original_sheets[
            "HTML매칭실패"
        ].to_excel(
            writer,
            sheet_name="HTML매칭실패",
            index=False
        )


# ============================================================
# 25. 결과 요약
# ============================================================

print()
print("=" * 70)
print("🎉 Gemini OCR 의미 해석 완료")
print("=" * 70)

print()
print(
    f"📊 이미지별 AI 해석 : "
    f"{len(image_ai_df):,}건"
)

print(
    f"📊 상품별 AI 통합 : "
    f"{len(product_ai_df):,}건"
)

print(
    f"⚠️ AI 오류/빈 OCR : "
    f"{len(error_df):,}건"
)

print()
print(
    f"📂 최종 파일 : "
    f"{OUTPUT_XLSX}"
)


# ============================================================
# 26. 결과 미리보기
# ============================================================

if len(product_ai_df) > 0:

    print()
    print("=" * 70)
    print("📋 상품 통합 결과 미리보기")
    print("=" * 70)

    preview_cols = [
        "통합제품ID",
        "브랜드",
        "상품명_Master",
        "AI상품명",
        "제조사",
        "수입원",
        "원재료",
        "마케팅소구포인트",
        "용량",
        "개당중량",
        "총중량",
        "수량",
        "포장당매수",
        "묶음수",
        "신뢰도",
        "처리상태"
    ]

    preview_cols = [
        c for c in preview_cols
        if c in product_ai_df.columns
    ]

    display(
        product_ai_df[
            preview_cols
        ].head(20)
    )


# ============================================================
# 27. 자동 다운로드
# ============================================================

print()
print("=" * 70)
print("⬇️ Excel 자동 다운로드")
print("=" * 70)

try:

    from google.colab import files

    files.download(
        OUTPUT_XLSX
    )

    print(
        "✅ 다운로드가 시작되었습니다."
    )

except Exception as e:

    print(
        f"⚠️ 자동 다운로드 실패: {e}"
    )

    print()
    print(
        "파일은 다음 위치에 저장되어 있습니다:"
    )

    print(
        OUTPUT_XLSX
    )

OCR Merge

In [ ]:
# ============================================================
# [CELL 8] AI_상품통합 → 제품분석 병합
# ============================================================

import os
import pandas as pd
from IPython.display import display
from google.colab import files


# ============================================================
# 설정
# ============================================================

BASE_DIR = "/content/coupang_ocr"

RESULT_DIR = os.path.join(
    BASE_DIR,
    "results"
)

GEMINI_FILE = os.path.join(
    RESULT_DIR,
    "coupang_ocr_gemini.xlsx"
)

# Cell 2에서 업로드한 원본 Excel
PRODUCT_FILE = excel_path

# 최종 파일
base_name = os.path.splitext(
    os.path.basename(PRODUCT_FILE)
)[0]

OUTPUT_FILE = os.path.join(
    RESULT_DIR,
    f"{base_name}_AI상품통합.xlsx"
)


# ============================================================
# 파일 확인
# ============================================================

print("=" * 70)
print("📦 AI 상품통합 → 제품분석 병합")
print("=" * 70)

print(
    f"\n상품 원본 : {PRODUCT_FILE}"
)

print(
    f"Gemini 결과 : {GEMINI_FILE}"
)


if not os.path.exists(PRODUCT_FILE):

    raise FileNotFoundError(
        f"❌ Cell 2의 원본 Excel을 찾을 수 없습니다.\n"
        f"{PRODUCT_FILE}"
    )


if not os.path.exists(GEMINI_FILE):

    raise FileNotFoundError(
        f"❌ Gemini 결과 파일을 찾을 수 없습니다.\n"
        f"{GEMINI_FILE}\n\n"
        "먼저 Cell 7을 실행해주세요."
    )


# ============================================================
# 1. 원본 상품 Excel 읽기
# ============================================================

product_xls = pd.ExcelFile(
    PRODUCT_FILE
)

print(
    "\n📄 원본 Excel 시트:",
    product_xls.sheet_names
)


if "제품분석" not in product_xls.sheet_names:

    raise ValueError(
        "❌ 원본 Excel에 '제품분석' 시트가 없습니다."
    )


product_df = pd.read_excel(
    PRODUCT_FILE,
    sheet_name="제품분석"
)

print(
    f"\n제품분석 상품 수 : {len(product_df):,}"
)


# ============================================================
# 2. Gemini 결과 읽기
# ============================================================

gemini_xls = pd.ExcelFile(
    GEMINI_FILE
)

print(
    "\n🤖 Gemini 결과 시트:",
    gemini_xls.sheet_names
)


if "AI_상품통합" not in gemini_xls.sheet_names:

    raise ValueError(
        "❌ Gemini 결과에 'AI_상품통합' 시트가 없습니다."
    )


ai_df = pd.read_excel(
    GEMINI_FILE,
    sheet_name="AI_상품통합"
)


print(
    f"AI_상품통합 행 수 : {len(ai_df):,}"
)


# ============================================================
# 3. ProductMaster 읽기
#    AI_상품통합에는 vendorItemId가 없으므로
#    통합제품ID → vendorItemId 연결
# ============================================================

if "ProductMaster" not in gemini_xls.sheet_names:

    raise ValueError(
        "❌ Gemini 결과에 'ProductMaster' 시트가 없습니다."
    )


pm_df = pd.read_excel(
    GEMINI_FILE,
    sheet_name="ProductMaster"
)


required_pm = [
    "통합제품ID",
    "vendorItemId"
]

missing_pm = [
    c for c in required_pm
    if c not in pm_df.columns
]

if missing_pm:

    raise ValueError(
        "❌ ProductMaster에 필요한 컬럼이 없습니다.\n"
        f"누락: {missing_pm}\n"
        f"현재 컬럼: {list(pm_df.columns)}"
    )


# ============================================================
# 4. ID 표준화
# ============================================================

def normalize_id(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Excel에서 숫자가 93506169634.0 형태로 읽힌 경우
    if value.endswith(".0"):

        try:

            value = str(
                int(float(value))
            )

        except:

            pass

    return value


product_df["vendorItemId"] = (
    product_df["vendorItemId"]
    .apply(normalize_id)
)

ai_df["통합제품ID"] = (
    ai_df["통합제품ID"]
    .apply(normalize_id)
)

pm_df["통합제품ID"] = (
    pm_df["통합제품ID"]
    .apply(normalize_id)
)

pm_df["vendorItemId"] = (
    pm_df["vendorItemId"]
    .apply(normalize_id)
)


# ============================================================
# 5. AI_상품통합에 vendorItemId 연결
# ============================================================

id_map = (
    pm_df[
        [
            "통합제품ID",
            "vendorItemId"
        ]
    ]
    .drop_duplicates(
        subset=["통합제품ID"]
    )
)


ai_df = ai_df.merge(
    id_map,
    on="통합제품ID",
    how="left"
)


# ============================================================
# 6. AI 컬럼 정의
# ============================================================

ai_columns = [
    "AI상품명",
    "구성품",
    "제조사",
    "수입원",
    "원재료",
    "알레르기",
    "마케팅소구포인트",
    "AI사이즈",
    "AI용량",
    "개당중량",
    "총중량_AI",
    "수량_AI",
    "AI포장당매수",
    "AI묶음수",
    "원산지",
    "포장재질",
    "보관방법",
    "소비기한",
    "사용방법",
    "기타정보",
    "근거이미지",
    "확인필요",
    "신뢰도",
    "AI이미지수",
    "AI처리상태"
]


# ============================================================
# 7. AI_상품통합 컬럼명 → 제품분석용 컬럼
# ============================================================

rename_map = {
    "상품명_Master": None,

    "상품명": "AI상품명",
    "AI상품명": "AI상품명",

    "구성품": "구성품",
    "제조사": "제조사",
    "수입원": "수입원",
    "원재료": "원재료",
    "알레르기": "알레르기",
    "마케팅소구포인트": "마케팅소구포인트",

    "사이즈": "AI사이즈",
    "용량": "AI용량",
    "개당중량": "개당중량",
    "총중량": "총중량_AI",

    "수량": "수량_AI",
    "포장당매수": "AI포장당매수",
    "묶음수": "AI묶음수",

    "원산지": "원산지",
    "포장재질": "포장재질",
    "보관방법": "보관방법",
    "소비기한": "소비기한",
    "사용방법": "사용방법",
    "기타정보": "기타정보",
    "근거이미지": "근거이미지",
    "확인필요": "확인필요",
    "신뢰도": "신뢰도",
    "이미지수": "AI이미지수",
    "처리상태": "AI처리상태"
}


# ============================================================
# 실제 존재하는 컬럼만 선택
# ============================================================

ai_merge = pd.DataFrame()

ai_merge["vendorItemId"] = ai_df[
    "vendorItemId"
]


for source_col, target_col in rename_map.items():

    if source_col == "상품명_Master":
        continue

    if source_col in ai_df.columns:

        ai_merge[target_col] = ai_df[
            source_col
        ]


# ============================================================
# AI상품명 처리
# ============================================================

# 실제 AI_상품통합에 AI상품명이 있으면 사용
if "AI상품명" in ai_df.columns:

    ai_merge["AI상품명"] = ai_df[
        "AI상품명"
    ]

# 혹시 AI상품명이 없고 상품명_Master가 있으면 사용
elif "상품명_Master" in ai_df.columns:

    ai_merge["AI상품명"] = ai_df[
        "상품명_Master"
    ]


# ============================================================
# 중복 제거
# ============================================================

ai_merge = (
    ai_merge
    .drop_duplicates(
        subset=["vendorItemId"],
        keep="first"
    )
)


# ============================================================
# 8. 기존 제품분석과 병합
# ============================================================

before_count = len(product_df)


merged_df = product_df.merge(
    ai_merge,
    on="vendorItemId",
    how="left",
    sort=False
)


# ============================================================
# 9. 원본 행 수 검증
# ============================================================

after_count = len(merged_df)


print("\n" + "=" * 70)
print("🔍 병합 검증")
print("=" * 70)

print(
    f"원본 제품분석 : {before_count:,}행"
)

print(
    f"병합 후       : {after_count:,}행"
)


if before_count != after_count:

    raise ValueError(
        "❌ 병합 후 행 수가 달라졌습니다.\n"
        "AI_상품통합의 vendorItemId 중복 여부를 확인해주세요."
    )


# ============================================================
# 10. 매칭 통계
# ============================================================

ai_match_count = (
    merged_df["AI처리상태"]
    .notna()
    .sum()
)

ai_unmatch_count = (
    len(merged_df)
    - ai_match_count
)


print(
    f"\n✅ AI 매칭 성공 : {ai_match_count:,}개"
)

print(
    f"⚠️ AI 매칭 없음 : {ai_unmatch_count:,}개"
)


# ============================================================
# 11. 매칭되지 않은 상품 확인
# ============================================================

if ai_unmatch_count > 0:

    unmatched = merged_df[
        merged_df["AI처리상태"].isna()
    ][
        [
            "vendorItemId",
            "브랜드",
            "상품명"
        ]
    ]

    print("\n⚠️ AI 매칭되지 않은 상품:")

    display(
        unmatched.head(20)
    )


# ============================================================
# 12. AI 컬럼 순서 정리
# ============================================================

# 실제 존재하는 AI 컬럼만
existing_ai_columns = [
    c
    for c in ai_columns
    if c in merged_df.columns
]


# 원본 컬럼
original_columns = [
    c
    for c in product_df.columns
    if c in merged_df.columns
]


# 최종 순서:
# 원본 제품분석 컬럼 → AI 분석 컬럼
final_columns = (
    original_columns
    + existing_ai_columns
)


merged_df = merged_df[
    final_columns
]


# ============================================================
# 13. 최종 Excel 저장
# ============================================================

print("\n💾 Excel 저장 중...")


# 원본 Excel의 모든 시트를 유지
with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    for sheet_name in product_xls.sheet_names:

        if sheet_name == "제품분석":

            merged_df.to_excel(
                writer,
                sheet_name="제품분석",
                index=False
            )

        else:

            temp_df = pd.read_excel(
                PRODUCT_FILE,
                sheet_name=sheet_name
            )

            temp_df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )


# ============================================================
# 14. 결과 확인
# ============================================================

print("\n" + "=" * 70)
print("🎉 CELL 8 완료")
print("=" * 70)

print(
    f"\n최종 파일 : {OUTPUT_FILE}"
)

print(
    f"상품 수   : {len(merged_df):,}"
)

print(
    f"전체 컬럼 : {len(merged_df.columns):,}"
)

print(
    "\n추가된 AI 컬럼:"
)

for col in existing_ai_columns:

    print(
        f"  • {col}"
    )


# ============================================================
# 15. 실제 AI 데이터 확인
# ============================================================

print("\n" + "=" * 70)
print("🔎 AI 상품통합 결과 샘플")
print("=" * 70)


preview_columns = [
    "vendorItemId",
    "브랜드",
    "상품명",
    "AI상품명",
    "구성품",
    "제조사",
    "수입원",
    "마케팅소구포인트",
    "수량_AI",
    "AI처리상태"
]

preview_columns = [
    c
    for c in preview_columns
    if c in merged_df.columns
]


display(
    merged_df[
        preview_columns
    ].head(10)
)


# ============================================================
# 16. 자동 다운로드
# ============================================================

files.download(
    OUTPUT_FILE
)